<a href="https://colab.research.google.com/github/bernardes7/Paredes/blob/main/notebooks/Between_Voices_Corpus_Analysis_from_JSONs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1) Environment Setup & Imports

In [ ]:
!pip install dtaidistance
%pip install EMD-signal
!pip -q install gdown


import glob
from pathlib import Path
from PyEMD import EMD
from dtaidistance import dtw
from google.colab import drive
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist, squareform
from scipy.ndimage import gaussian_filter1d
import seaborn as sns
import itertools


#drive.mount('/content/drive')

# 2) Load Data (JSON)

In [ ]:

# Config: set your shared folder URL and local destination
DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1aKt0FP2U8ONcRo0cXrrYDdiqnuM9YkrZ?usp=sharing'
DEST_DIR = Path('/content/Results')
DEST_DIR.mkdir(parents=True, exist_ok=True)

# Download the shared folder into /content/Results
!gdown --fuzzy --folder "{DRIVE_FOLDER_URL}" -O "{DEST_DIR}"

# Find the actual downloaded folder (Drive folder name)
downloaded_root = DEST_DIR

print(f"Downloaded into: {downloaded_root}")

# Helper: load all JSON files recursively
def load_json_files(root_dir: Path):
    all_data = []
    json_paths = list(root_dir.rglob('*.json'))
    if not json_paths:
        print(f"No JSON files found under {root_dir}")
    for jp in json_paths:
        try:
            with open(jp, 'r', encoding='utf-8') as f:
                all_data.append(json.load(f))
        except json.JSONDecodeError as e:
            print(f"Error decoding {jp}: {e}")
        except Exception as e:
            print(f"Error reading {jp}: {e}")
    return all_data

data = load_json_files(downloaded_root)

print(f"Loaded {len(data)} JSON objects.")

# 3) Apply EMD

In [ ]:
import numpy as np
from PyEMD import EMD
import warnings

# Feature configuration
features_config_emd = [
    ('rhythmic', ['tempo_deviations'], 'Tempo deviation'),
    ('rhythmic', ['voice_rhythmic_density'], 'Voice rhythmic density'),
    ('rhythmic', ['guitars_rhythmic_density'], 'Gtr rhythmic density'),
    ('dynamic', ['voice_loudness'], 'Voice loudness'),
    ('dynamic', ['guitars_loudness'], 'Gtr loudness'),
    ('harmonic', ['tonal_dissonance'], 'Tonal dissonance'),
    ('harmonic', ['tonal_dispersion'], 'Tonal dispersion'),
    ('melodic', ['voice_melodic_contour'], 'Melodic voice contour'),
    ('melodic', ['guitars_harmonic_contour'], 'Harmonic gtr contour')
]

data_processed = []

for row_idx, row in enumerate(data):
    processed_row = {
        'metadata': row.get('metadata', {}),
        'features': {}
    }

    for domain, keys, label in features_config_emd:
        if domain not in row:
            warnings.warn(f"Row {row_idx}: Domain '{domain}' not found.")
            continue

        for key in keys:
            if key not in row[domain]:
                warnings.warn(f"Row {row_idx}: Feature '{key}' not found in domain '{domain}'.")
                continue

            values = row[domain].get(key, [])
            if values and isinstance(values, list) and all(isinstance(v, (int, float)) for v in values):
                signal = np.array(values)
                if np.all(signal == signal[0]):  # Skip constant signals
                    continue
                try:
                    emd = EMD()
                    imfs = emd(signal)
                    if imfs.shape[0] > 1:
                        processed_signal = np.sum(imfs[2:], axis=0).tolist()
                    else:
                        processed_signal = signal.tolist()
                    processed_row['features'][label] = processed_signal
                except Exception as e:
                    warnings.warn(f"Row {row_idx}: EMD failed for '{label}' ({e}). Using original signal.")
                    processed_row['features'][label] = signal.tolist()
            else:
                warnings.warn(f"Row {row_idx}: Invalid or empty values for feature '{key}' in domain '{domain}'.")

    data_processed.append(processed_row)

# 4) Apply Loudness Masks

In [ ]:
# Define Loudness Masks for Voice and Guitars
# Stored in the original raw 'data' as v_mark and g_mask

loudness_threshold = -40

for row in data:
    # Get original loudness arrays
    loudness_voice_orig = row.get('dynamic', {}).get('voice_loudness', [])
    loudness_guitars_orig = row.get('dynamic', {}).get('guitars_loudness', [])

    # Compute masks from original signals
    voice_mask = [val >= loudness_threshold if val is not None else False for val in loudness_voice_orig]
    guitar_mask = [val >= loudness_threshold if val is not None else False for val in loudness_guitars_orig]

    # Store masks for later use (e.g., in row or separate dict)
    row['dynamic']['v_mask'] = voice_mask
    row['dynamic']['g_mask'] = guitar_mask

print("Loudness masks computed and stored from original signals.")


for original_row, processed_row in zip(data, data_processed):
    # Get masks from original data
    voice_mask = original_row.get('dynamic', {}).get('v_mask', [])
    guitar_mask = original_row.get('dynamic', {}).get('g_mask', [])

    title = processed_row.get('metadata', {}).get('title', 'Unknown')
    artist = processed_row.get('metadata', {}).get('artist', 'Unknown')

    print(f"\n=== Applying Masks for Piece: {title} by {artist} ===")

    # Skip if both masks are empty
    if not voice_mask and not guitar_mask:
        print("⚠️ No masks available for this piece.")
        continue

    for feature_name, values in processed_row['features'].items():
        if isinstance(values, list) and values:  # Only process non-empty lists
            masked_values = None

            # Apply voice mask
            if ('Voice' in feature_name or 'Vocals' in feature_name) and voice_mask and len(values) == len(voice_mask):
                masked_values = [val if mask else np.nan for val, mask in zip(values, voice_mask)]

            # Apply guitar mask
            elif ('Guitars' in feature_name or 'Harmonic' in feature_name) and guitar_mask and len(values) == len(guitar_mask):
                masked_values = [val if mask else np.nan for val, mask in zip(values, guitar_mask)]

            # Update and print if masking applied
            if masked_values is not None:
                processed_row['features'][feature_name] = masked_values
                print(f"Feature: {feature_name} → masked values: {masked_values[:20]}{'...' if len(masked_values) > 20 else ''}")

In [ ]:
def get_processed_song(data_processed, title):
    title = title.lower()
    for row in data_processed:
        if title in row.get('metadata', {}).get('title', '').lower():
            return row['features']
    return None

# Usage:
features = get_processed_song(data_processed, "Não Choro por me Deixares")
if features:
    print(f"\n✅ Processed features for 'Não Choro por me Deixares':")
    for name, vals in features.items():
        if isinstance(vals, list):
            print(f"{name}: length={len(vals)}, values={vals[:10]}{'...' if len(vals) > 10 else ''}")
else:
    print("Song not found or no processed features.")

# 5) Descriptive Statistics

In [ ]:
import numpy as np
import pandas as pd

song_stats_raw = []

for row in data:
    title = row.get('metadata', {}).get('title', 'Unknown')
    artist = row.get('metadata', {}).get('artist', 'Unknown Artist')
    year = row.get('metadata', {}).get('project_year', None)

    # Convert year to numeric or NaN
    try:
        year = int(year)
    except (ValueError, TypeError):
        year = np.nan

    # Loop through features_config_emd (3 elements per tuple)
    for feature_category, feature_key_path, feature_name in features_config_emd:
        # Navigate nested dict without helper
        nested_data = row.get(feature_category, {})
        for key in feature_key_path:
            nested_data = nested_data.get(key, {})

        # Ensure it's a list of numeric values
        feature_values_raw = nested_data if isinstance(nested_data, list) else []
        numeric_values = [v for v in feature_values_raw if isinstance(v, (int, float)) and np.isfinite(v)]

        if numeric_values:
            song_stats_raw.append({
                'title': title,
                'artist': artist,
                'year': year,
                'feature': feature_name,
                'mean': np.mean(numeric_values),
                'std': np.std(numeric_values)
            })

# Build DataFrame if data exists
if song_stats_raw:
    df_song_stats_raw = pd.DataFrame(song_stats_raw)

    # Combine mean and std into one column
    df_song_stats_raw['mean_std'] = df_song_stats_raw.apply(
        lambda row: f"{row['mean']:.2f} ({row['std']:.2f})", axis=1
    )

    # Pivot table
    df_final_raw = df_song_stats_raw.pivot_table(
        index=['title', 'artist', 'year'],
        columns='feature',
        values='mean_std',
        aggfunc='first'
    )

    # Sort by year (NaN goes last)
    df_final_raw = df_final_raw.reset_index().sort_values(by='year', na_position='last').set_index(['title', 'artist', 'year'])

    display(df_final_raw)
else:
    print("No raw feature data available to calculate song statistics.")

# 6) Pearson Correlation

In [ ]:

# -*- coding: utf-8 -*-
"""
Fisher's z-transform aggregation (signed) for correlation matrices
- Per-piece correlations
- Artist-level aggregation using arctanh/tanh with optional weighting (n-3)
- Global aggregation (across all pieces)
- NaN-safe aggregation per cell
- Matrix alignment to common feature set
"""

import os
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =========================
# Configuration parameters
# =========================
CORR_METHOD = 'pearson'  # 'pearson' or 'spearman'
USE_WEIGHTING = True     # If True, weight Fisher z means by (n - 3)
PLOT_PER_PIECE = True    # Plot per-piece correlation heatmaps
SAVE_OUTPUT = False      # Save artist/global aggregated matrices to disk
OUTPUT_DIR = "outputs"   # Directory for saved files if SAVE_OUTPUT=True

# --- Graphical Parameters (New Variables) ---
ANNOT_SIZE_PER_PIECE = 13 # Increased for consistency
FIGSIZE_PER_PIECE = (8, 7)

ANNOT_SIZE_ARTIST = 16
FIGSIZE_ARTIST = (10, 9) # Increased height to 9 as per previous request

ANNOT_SIZE_GLOBAL = 16
FIGSIZE_GLOBAL = (10, 9) # Increased height to 9 as per previous request
# --------------------------------------------

# =========================
# Utility functions
# =========================

def align_matrices_to_common_features(matrices, master_feature_order):
    """
    Align a list of correlation matrices to a master feature order.
    Missing features in individual matrices will be filled with NaN.
    If the master_feature_order has fewer than 2 features, returns an empty list.
    """
    if not matrices or len(master_feature_order) < 2:
        return []

    aligned = []
    for m in matrices:
        # Reindex to master_feature_order, filling missing with NaN
        reindexed_m = m.reindex(index=master_feature_order, columns=master_feature_order)
        aligned.append(reindexed_m)
    return aligned


def fisher_z_aggregate_signed(matrices, sample_sizes=None, eps=1e-12):
    """
    Aggregate a list of correlation matrices using Fisher's z-transform,
    preserving directionality (signed correlations).

    Parameters
    ----------
    matrices : List[pd.DataFrame]
        All matrices must be square and share the same shape/index/columns.
    sample_sizes : List[int] or None
        Optional per-piece n used to weight Fisher z means by (n - 3).
        If None, unweighted mean is used.
    eps : float
        Small value to clip r in (-1+eps, 1-eps) to avoid arctanh infinities.

    Returns
    -------
    pd.DataFrame
        Aggregated correlation matrix (signed, may contain NaNs if all inputs NaN for a cell).
    """
    if not matrices:
        raise ValueError("No matrices to aggregate.")

    # Stack as K x P x P
    stacked = np.stack([m.values for m in matrices])  # shape: K x P x P
    K, P, Q = stacked.shape
    if P != Q:
        raise ValueError("Correlation matrices must be square.")

    # Clip to avoid arctanh(±1) -> inf
    stacked = np.clip(stacked, -1 + eps, 1 - eps)

    # Fisher z-transform
    z = np.arctanh(stacked)  # K x P x P

    # Prepare weights
    if sample_sizes is not None:
        if len(sample_sizes) != K:
            raise ValueError("sample_sizes length must match number of matrices.")
        w = np.array(sample_sizes, dtype=float) - 3.0
        # Non-negative weights (for n <= 3, set weight 0)
        w = np.maximum(w, 0.0)
        use_weights = True
    else:
        use_weights = False
        w = np.ones(K, dtype=float)

    # NaN-safe cellwise weighted mean in z-space
    z_mean = np.empty((P, P), dtype=float)
    z_mean[:] = np.nan

    for i in range(P):
        for j in range(P):
            z_ij = z[:, i, j]                 # K values for this cell
            mask = np.isfinite(z_ij)          # valid entries
            if not np.any(mask):
                continue                      # remains NaN if all invalid
            if use_weights:
                w_valid = w[mask]
                denom = w_valid.sum()
                if denom > 0:
                    z_mean[i, j] = np.dot(w_valid, z_ij[mask]) / denom
                else:
                    z_mean[i, j] = np.nanmean(z_ij[mask])     # fallback
            else:
                z_mean[i, j] = np.nanmean(z_ij[mask])

    # Back-transform to r (tanh)
    r_mean = np.tanh(z_mean)

    # Return with original labels from the first matrix
    return pd.DataFrame(r_mean, index=matrices[0].index, columns=matrices[0].columns)


def plot_heatmap(df, title, vmin=-1, vmax=1, annot=True, annot_size=12, figsize=(10, 8)):
    """Convenience wrapper for seaborn heatmap with consistent styling."""
    plt.figure(figsize=figsize)
    ax = sns.heatmap(df, annot=annot, cmap='coolwarm', fmt=".2f",
                vmin=vmin, vmax=vmax, annot_kws={"size": annot_size},
                cbar_kws={'shrink': 0.8})
    plt.title(title, pad=20, fontsize=annot_size) # Apply fontsize to title

    # Removed explicit plt.xlabel and plt.ylabel calls
    # plt.xlabel('Features', fontsize=annot_size)
    # plt.ylabel('Features', fontsize=annot_size)

    # Set fontsize for tick labels (feature names) directly on the axes
    ax.tick_params(axis='x', labelsize=annot_size)
    ax.tick_params(axis='y', labelsize=annot_size)

    plt.tight_layout()
    plt.show()


def ensure_output_dir(path):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)


# =========================
# Main pipeline
# =========================
# NOTE: Expects `data_processed` defined elsewhere (list of dicts).
# Each item structure example:
# {
#   'metadata': {'artist': 'Artist Name', 'title': 'Piece Title'},
#   'features': {'featA': [...], 'featB': [...], 'Voice Mask': [...], ...}
# }

def run_pipeline(data_processed):
    master_feature_order = []
    first_piece_feature_order_established = False

    # Sort and group by artist (required for itertools.groupby)
    data_sorted_by_artist = sorted(
        data_processed,
        key=lambda x: x.get('metadata', {}).get('artist', 'Unknown Artist')
    )
    artist_groups = itertools.groupby(
        data_sorted_by_artist,
        key=lambda x: x.get('metadata', {}).get('artist', 'Unknown Artist')
    )

    # First pass to establish master_feature_order from the first encountered piece
    # and collect all unique feature names
    all_unique_features = set()
    for row in data_processed:
        features = row.get('features', {})
        current_piece_features = [k for k in features.keys() if k not in ['Voice Mask', 'Guitar Mask']]

        if not first_piece_feature_order_established and len(current_piece_features) >= 2:
            master_feature_order = list(current_piece_features) # Establish order from the first valid piece
            first_piece_feature_established = True

        for k in current_piece_features:
            all_unique_features.add(k)

    # If master_feature_order was not established from the first piece (e.g., first piece had < 2 features)
    # establish it from the sorted list of all unique features found across the corpus.
    if not first_piece_feature_established:
        if len(all_unique_features) >= 2:
            master_feature_order = sorted(list(all_unique_features))
        else:
            print("Fewer than 2 unique features found across the entire dataset. Skipping pipeline.")
            return []
    else: # Append any features not in the first piece's order to the end of master_feature_order
        for feature in sorted(list(all_unique_features)):
            if feature not in master_feature_order:
                master_feature_order.append(feature)

    if len(master_feature_order) < 2:
        print("Fewer than 2 features available for correlation analysis. Skipping pipeline.")
        return []

    global_matrices = []
    global_ns = []  # per-piece sample sizes (n) for weighting
    artist_correlation_matrices = []  # list of (artist, aggregated_df)

    sns.set(style="whitegrid")

    # Reset artist_groups iterator as we've already iterated once
    data_sorted_by_artist = sorted(
        data_processed,
        key=lambda x: x.get('metadata', {}).get('artist', 'Unknown Artist')
    )
    artist_groups = itertools.groupby(
        data_sorted_by_artist,
        key=lambda x: x.get('metadata', {}).get('artist', 'Unknown Artist')
    )

    for artist, pieces in artist_groups:
        artist_pieces = list(pieces)
        piece_mats = []
        piece_ns = []

        for row in artist_pieces:
            title = row.get('metadata', {}).get('title', 'Unknown')
            features = row.get('features', {})

            # Filter: keep only numeric lists; drop masks
            piece_data = {
                k: v for k, v in features.items()
                if isinstance(v, list) and k not in ['Voice Mask', 'Guitar Mask'] and len(v) > 0
            }

            # Need at least 2 features to compute correlations
            if len(piece_data) < 2:
                # print(f"⚠️ Skipping {title} by {artist}: less than 2 numeric features.")
                continue

            # Ensure all features have equal length
            lengths = {len(vals) for vals in piece_data.values()}
            if len(lengths) != 1:
                # print(f"⚠️ Skipping {title} by {artist}: features have different lengths.")
                continue

            n = next(iter(lengths))  # sample size per piece
            if n < 2:
                # print(f"⚠️ Skipping {title} by {artist}: insufficient samples (n={n}).")
                continue

            # Build DataFrame and compute correlation
            df_piece = pd.DataFrame(piece_data)
            corr_piece = df_piece.corr(method=CORR_METHOD)

            # Align this piece's matrix to the master_feature_order
            aligned_corr_piece = align_matrices_to_common_features([corr_piece], master_feature_order)[0]

            piece_mats.append(aligned_corr_piece)
            piece_ns.append(n)

            global_matrices.append(aligned_corr_piece)
            global_ns.append(n)

            # Optional per-piece plot (using the aligned matrix for consistent labels)
            if PLOT_PER_PIECE:
                plot_heatmap(
                    aligned_corr_piece,
                    title=f"Correlation Matrix - {title} ({artist})",
                    vmin=-1, vmax=1, annot=True, annot_size=ANNOT_SIZE_PER_PIECE, figsize=FIGSIZE_PER_PIECE
                )

        # Artist-level aggregation with Fisher z (signed)
        if piece_mats:
            # piece_mats are already aligned to master_feature_order
            if USE_WEIGHTING:
                artist_matrix_signed = fisher_z_aggregate_signed(
                    piece_mats, sample_sizes=piece_ns
                )
            else:
                artist_matrix_signed = fisher_z_aggregate_signed(
                    piece_mats, sample_sizes=None
                )

            artist_correlation_matrices.append((artist, artist_matrix_signed))

            plot_heatmap(
                artist_matrix_signed,
                title=f"Correlation Matrix - {artist}",
                vmin=-1, vmax=1, annot=True, annot_size=ANNOT_SIZE_ARTIST, figsize=FIGSIZE_ARTIST
            )

            # Optional save
            if SAVE_OUTPUT:
                ensure_output_dir(OUTPUT_DIR)
                # Save CSV and XLSX
                csv_path = os.path.join(OUTPUT_DIR, f"{artist}_fisherz_signed_{CORR_METHOD}.csv")
                xlsx_path = os.path.join(OUTPUT_DIR, f"{artist}_fisherz_signed_{CORR_METHOD}.xlsx")
                artist_matrix_signed.to_csv(csv_path, index=True)
                artist_matrix_signed.to_excel(xlsx_path, engine='openpyxl')

    # Global aggregation across all pieces
    if global_matrices:
        # global_matrices are already aligned to master_feature_order
        if USE_WEIGHTING:
            global_matrix_signed = fisher_z_aggregate_signed(
                global_matrices, sample_sizes=global_ns
            )
        else:
            global_matrix_signed = fisher_z_aggregate_signed(
                global_matrices, sample_sizes=None
            )

        plot_heatmap(
            global_matrix_signed,
            title="Global Correlation Matrix (All Pieces)",
            vmin=-1, vmax=1, annot=True, annot_size=ANNOT_SIZE_GLOBAL, figsize=FIGSIZE_GLOBAL
        )

        # Optional save
        if SAVE_OUTPUT:
            ensure_output_dir(OUTPUT_DIR)
            csv_path = os.path.join(OUTPUT_DIR, f"GLOBAL_fisherz_signed_{CORR_METHOD}.csv")
            xlsx_path = os.path.join(OUTPUT_DIR, f"GLOBAL_fisherz_signed_{CORR_METHOD}.xlsx")
            global_matrix_signed.to_csv(csv_path, index=True)
            global_matrix_signed.to_excel(xlsx_path, engine='openpyxl')

    return artist_correlation_matrices  # list of (artist, aggregated_df)


# =========================
# Example call (uncomment)
# =========================
results = run_pipeline(data_processed)
for artist, mat in results:
    print(f"\n=== {artist} ===")


# 7) Mutual Information

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from sklearn.feature_selection import mutual_info_regression

# Sort and group processed data by artist
data_sorted_by_artist = sorted(data_processed, key=lambda x: x.get('metadata', {}).get('artist', 'Unknown Artist'))
artist_groups = itertools.groupby(data_sorted_by_artist, key=lambda x: x.get('metadata', {}).get('artist', 'Unknown Artist'))

global_mi_matrices = []

for artist, pieces in artist_groups:
    artist_pieces = list(pieces)
    piece_mi_matrices_artist = {}

    for row in artist_pieces:
        title = row.get('metadata', {}).get('title', 'Unknown')
        features = row.get('features', {})

        # Filter out masks and keep only numeric features
        piece_data = {k: v for k, v in features.items()
                      if isinstance(v, list) and k not in ['Voice Mask', 'Guitar Mask'] and len(v) > 0}

        # Ensure all features have equal length
        if len(piece_data) > 1:
            lengths = {len(vals) for vals in piece_data.values()}
            if len(lengths) == 1:  # All equal length
                df_piece = pd.DataFrame(piece_data)

                # Initialize MI matrix
                mi_matrix = pd.DataFrame(np.zeros((df_piece.shape[1], df_piece.shape[1])),
                                         index=df_piece.columns,
                                         columns=df_piece.columns)

                # Compute Mutual Information for each pair
                for i, col_i in enumerate(df_piece.columns):
                    for j, col_j in enumerate(df_piece.columns):
                        if i != j:
                            valid_rows = df_piece[[col_i, col_j]].dropna()
                            if valid_rows.shape[0] > 1:
                                mi = mutual_info_regression(valid_rows[[col_i]], valid_rows[col_j], discrete_features=False)[0]
                                mi_matrix.iloc[i, j] = mi

                piece_mi_matrices_artist[title] = mi_matrix
                global_mi_matrices.append(mi_matrix)

                # Plot per-piece MI heatmap
                plt.figure(figsize=(8, 6))
                sns.heatmap(mi_matrix, annot=True, cmap='viridis', fmt=".2f")
                plt.title(f"Mutual Information Matrix - {title} by {artist}")
                plt.tight_layout()
                plt.show()
            else:
                print(f"⚠️ Skipping {title}: features have different lengths.")

    # Artist-level median MI matrix
    if piece_mi_matrices_artist:
        stacked = np.stack([m.values for m in piece_mi_matrices_artist.values()])
        median_matrix = np.median(stacked, axis=0)
        artist_mi_matrix = pd.DataFrame(median_matrix,
                                        index=piece_mi_matrices_artist[next(iter(piece_mi_matrices_artist))].index,
                                        columns=piece_mi_matrices_artist[next(iter(piece_mi_matrices_artist))].columns)

        plt.figure(figsize=(10, 8))
        sns.heatmap(artist_mi_matrix, annot=True, cmap='viridis', fmt=".2f")
        plt.title(f"Median Mutual Information Matrix - {artist}")
        plt.tight_layout()
        plt.show()

# Global median MI matrix
if global_mi_matrices:
    stacked_global = np.stack([m.values for m in global_mi_matrices])
    median_global = np.median(stacked_global, axis=0)
    global_mi_matrix = pd.DataFrame(median_global,
                                    index=global_mi_matrices[0].index,
                                    columns=global_mi_matrices[0].columns)

    plt.figure(figsize=(10, 8))
    sns.heatmap(global_mi_matrix, annot=True, cmap='viridis', fmt=".2f")
    plt.title("Global Median Mutual Information Matrix (All Pieces)")
    plt.tight_layout()
    plt.show()

# 8) Phrase Clustering

In [ ]:
def get_nested_value(d, keys):
    """Safely retrieve nested values from a dictionary using a list of keys."""
    for key in keys:
        if isinstance(d, dict):
            d = d.get(key, {})
        else:
            return []
    return d if isinstance(d, list) else []

def analyze_feature(data, feature_category, feature_key_path, time_key, time_category):
    phrase_data = []
    for row in data:
        title = row.get('metadata', {}).get('title', 'Unknown')
        feature_values = get_nested_value(row.get(feature_category, {}), feature_key_path)
        time_values = row.get(time_category, {}).get(time_key, [])
        structural = row.get('structural', {})
        phrase_labels = structural.get('phrase_labels', [])
        phrase_times = structural.get('phrase_times', [])
        phrase_durations = structural.get('phrase_durations', [])

        if not (len(phrase_labels) == len(phrase_times) == len(phrase_durations)):
            continue
        if not (isinstance(time_values, list) and isinstance(feature_values, list)):
            continue

        for label, start, duration in zip(phrase_labels, phrase_times, phrase_durations):
            end = start + duration
            # Extract all values within the phrase's time window, including NaNs
            segment_values = []
            for t, v in zip(time_values, feature_values):
                if start - 1e-9 <= t <= end + 1e-9:
                    segment_values.append(v)

            # Convert to numpy array for consistent handling later (e.g., NaN checks)
            values_np = np.array(segment_values, dtype=float)
            phrase_data.append({'title': title, 'phrase_label': label, 'values': values_np})

    # Filter out entries where 'values' was empty after extraction (e.g., no time points matched)
    df_phrases = pd.DataFrame([p for p in phrase_data if len(p['values']) > 0])
    if df_phrases.empty:
        print(f"No valid phrase data found for {'/'.join(feature_key_path)}")
        return pd.DataFrame()

    # Determine max_len based on initial extracted sequence lengths
    max_len = df_phrases['values'].apply(len).max()
    if max_len == 0:
        print(f"Max length is 0 for {'/'.join(feature_key_path)}, returning empty DataFrame.")
        return pd.DataFrame()

    def interpolate_and_smooth_and_normalize(sequences_of_np_arrays, target_length, sigma=1):
        interpolated_smoothed_normalized = []
        for seq_np in sequences_of_np_arrays: # seq_np is already a numpy array, potentially with NaNs
            valid_count = np.count_nonzero(~np.isnan(seq_np))
            total_count = len(seq_np)

            # Handle empty sequences or sequences that are mostly NaNs (less than 80% valid)
            if total_count == 0 or valid_count == 0 or (valid_count / total_count < 0.8 and total_count > 0):
                interpolated_smoothed_normalized.append(np.zeros(target_length).tolist())
                continue

            # Step 1: Interpolate NaNs using pandas Series interpolate (handles leading/trailing NaNs)
            interpolated_seq_pre_resample = seq_np.copy() # Make a copy to avoid modifying original array
            if valid_count >= 2: # Need at least 2 non-NaN points for linear interpolation to work effectively
                interpolated_seq_pre_resample = pd.Series(interpolated_seq_pre_resample).interpolate(method='linear', limit_direction='both').values
                # If there are still NaNs (e.g., only one non-NaN point and limit_direction couldn't fill),
                # fill remaining with the mean of non-NaN values or 0 if no valid points (should be caught by valid_count check)
                if np.isnan(interpolated_seq_pre_resample).any():
                    if valid_count > 0: # This case covers when valid_count >= 2 but interpolate couldn't fill all (e.g., all same value)
                        fill_value = np.nanmean(interpolated_seq_pre_resample)
                        interpolated_seq_pre_resample = np.nan_to_num(interpolated_seq_pre_resample, nan=fill_value)
                    else:
                        interpolated_seq_pre_resample = np.zeros_like(interpolated_seq_pre_resample)
            elif valid_count == 1: # If only one valid point, fill the entire sequence with that point
                interpolated_seq_pre_resample = np.full_like(seq_np, seq_np[~np.isnan(seq_np)][0])
            # If valid_count is 0, it's handled by the earlier 'if' condition.


            # Step 2: Resample, Smooth, and Normalize
            if len(interpolated_seq_pre_resample) >= 2:
                x_old = np.arange(len(interpolated_seq_pre_resample))
                f_interp = interp1d(x_old, interpolated_seq_pre_resample, kind='linear', fill_value="extrapolate")
                x_new = np.linspace(0, len(interpolated_seq_pre_resample) - 1, target_length)
                resampled_seq = f_interp(x_new)
                smoothed = gaussian_filter1d(resampled_seq, sigma=sigma)

                mean = np.mean(smoothed)
                std = np.std(smoothed)
                if std != 0:
                    normalized = (smoothed - mean) / std
                else:
                    normalized = smoothed - mean # Center but don't scale if std is 0
                interpolated_smoothed_normalized.append(normalized.tolist())
            elif len(interpolated_seq_pre_resample) == 1:
                # If after interpolation, there's effectively one point, normalize it to 0 and fill target_length
                interpolated_smoothed_normalized.append(np.zeros(target_length).tolist()) # Center to 0 for a flat line
            else: # Empty sequence after all steps (should be caught by previous conditions)
                interpolated_smoothed_normalized.append(np.zeros(target_length).tolist())

        return interpolated_smoothed_normalized


    df_phrases['values'] = interpolate_and_smooth_and_normalize(df_phrases['values'].tolist(), max_len, sigma=1)
    return df_phrases


def plot_feature_clusters_cosine(df_feature, feature_name):
    if df_feature.empty or 'values' not in df_feature.columns:
        print(f"Skipping {feature_name}: missing data.")
        return

    print(f"Plotting dendrogram and clusters for {feature_name} (Cosine)...")
    sequences = df_feature['values'].tolist()
    array_sequences = np.array(sequences)

    if array_sequences.ndim != 2:
        print(f"Error: Expected 2D array for clustering, got shape {array_sequences.shape}")
        return

    valid_indices = [i for i, seq in enumerate(array_sequences) if np.linalg.norm(seq) > 0]
    if len(valid_indices) < 2:
        print(f"Skipping {feature_name}: not enough valid sequences for cosine distance.")
        return

    array_sequences = array_sequences[valid_indices]
    df_feature = df_feature.iloc[valid_indices].reset_index(drop=True)

    distance_matrix = pdist(array_sequences, metric='cosine')
    linked = linkage(distance_matrix, method='average')

    heights = linked[:, 2]
    height_diffs = np.diff(heights)
    threshold = np.percentile(height_diffs, 98)
    split_indices = np.where(height_diffs > threshold)[0]
    estimated_clusters = len(array_sequences) - split_indices[0] if len(split_indices) > 0 else 5

    clusters = fcluster(linked, t=estimated_clusters, criterion='maxclust')
    df_feature['cluster'] = clusters

    # Ajuste do color_threshold para corresponder ao número de clusters
    if estimated_clusters < len(linked) + 1:
        color_threshold = linked[-estimated_clusters + 1, 2]
    else:
        color_threshold = linked[-1, 2] + 1

    combined_labels = [f"{row['title']}_{row['phrase_label']}" for index, row in df_feature.iterrows()]

    plt.figure(figsize=(12, 6))
    plt.title(f"Dendrogram - {feature_name} (Clusters: {estimated_clusters})")
    dendrogram(linked,
               labels=combined_labels,
               leaf_rotation=90,
               leaf_font_size=7,
               color_threshold=color_threshold)
    plt.xlabel("Phrase Label")
    plt.ylabel("Cosine Distance")
    plt.tight_layout()
    plt.show()

    # Plot clusters com 10 ou mais sequências
    for cluster_id in sorted(df_feature['cluster'].unique()):
        cluster_df = df_feature[df_feature['cluster'] == cluster_id]
        if len(cluster_df) < 10:
            continue
        cluster_sequences = cluster_df['values'].tolist()
        cluster_cosine_distances = squareform(pdist(np.array(cluster_sequences), metric='cosine'))
        total_distances = np.sum(cluster_cosine_distances, axis=1)
        medoid_index = np.argmin(total_distances)
        medoid_sequence = np.array(cluster_sequences[medoid_index])

        plt.figure(figsize=(10, 5))
        for seq in cluster_sequences:
            plt.plot(seq, color='lightgray', linewidth=0.5)
        plt.plot(medoid_sequence, color='red', linewidth=2, label='Medoid')
        plt.title(f"{feature_name} - Cluster {cluster_id} (n={len(cluster_df)})")
        plt.xlabel("Interpolated Index")
        plt.ylabel("Feature Value (Normalized)")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        print(f"\nSequences in {feature_name} - Cluster {cluster_id}:")
        for index, row in cluster_df.iterrows():
            print(f"- {row['title']}_{row['phrase_label']}")


def plot_feature_clusters_dtw(df_feature, feature_name):
    if df_feature.empty or 'values' not in df_feature.columns:
        print(f"Skipping {feature_name}: missing data.")
        return

    print(f"Plotting dendrogram and clusters for {feature_name} (DTW)...")
    sequences = df_feature['values'].tolist()
    array_sequences = np.array(sequences)

    if array_sequences.ndim != 2:
        print(f"Error: Expected 2D array for clustering, got shape {array_sequences.shape}")
        return
    dtw_distances = np.zeros((len(sequences), len(sequences)))
    for i in range(len(sequences)):
        for j in range(i + 1, len(sequences)):
            dist = dtw.distance(sequences[i], sequences[j])
            dtw_distances[i, j] = dist
            dtw_distances[j, i] = dist

    linked = linkage(squareform(dtw_distances), method='average')

    heights = linked[:, 2]
    height_diffs = np.diff(heights)
    threshold = np.percentile(height_diffs, 98)
    split_indices = np.where(height_diffs > threshold)[0]
    estimated_clusters = len(sequences) - split_indices[0] if len(split_indices) > 0 else 5

    clusters = fcluster(linked, t=estimated_clusters, criterion='maxclust')
    df_feature['cluster'] = clusters

    # Ajuste do color_threshold para corresponder ao número de clusters
    if estimated_clusters < len(linked) + 1:
        color_threshold = linked[-estimated_clusters + 1, 2]
    else:
        color_threshold = linked[-1, 2] + 1

    combined_labels = [f"{row['title']}_{row['phrase_label']}" for index, row in df_feature.iterrows()]

    plt.figure(figsize=(12, 6))
    plt.title(f"DTW Dendrogram - {feature_name} (Clusters: {estimated_clusters})")
    dendrogram(linked,
               labels=combined_labels,
               leaf_rotation=90,
               leaf_font_size=7,
               color_threshold=color_threshold)
    plt.xlabel("Phrase Label")
    plt.ylabel("DTW Distance")
    plt.tight_layout()
    plt.show()

    # Plot clusters com 10 ou mais sequências
    for cluster_id in sorted(df_feature['cluster'].unique()):
        cluster_df = df_feature[df_feature['cluster'] == cluster_id]
        if len(cluster_df) < 10:
            continue
        cluster_sequences = cluster_df['values'].tolist()
        cluster_dtw_distances = np.zeros((len(cluster_sequences), len(cluster_sequences)))
        for i in range(len(cluster_sequences)):
            for j in range(i + 1, len(cluster_sequences)):
                dist = dtw.distance(cluster_sequences[i], cluster_sequences[j])
                cluster_dtw_distances[i, j] = dist
                cluster_dtw_distances[j, i] = dist

        total_distances = np.sum(cluster_dtw_distances, axis=1)
        medoid_index = np.argmin(total_distances)
        medoid_sequence = np.array(cluster_sequences[medoid_index])

        plt.figure(figsize=(10, 5))
        for seq in cluster_sequences:
            plt.plot(seq, color='lightgray', linewidth=0.5)
        plt.plot(medoid_sequence, color='red', linewidth=2, label='Medoid')
        plt.title(f"{feature_name} - DTW Cluster {cluster_id} (n={len(cluster_df)})")
        plt.xlabel("Interpolated Index")
        plt.ylabel("Feature Value (Normalized)")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        print(f"\nSequences in {feature_name} - DTW Cluster {cluster_id}:")
        for index, row in cluster_df.iterrows():
            print(f"- {row['title']}_{row['phrase_label']}")

method_selector = 'cosine'  # ou 'dtw'

features_config = [
    ('rhythmic', ['tempo_deviations'], 'beat_times', 'rhythmic', 'Tempo Deviations'),
    ('rhythmic', ['guitars_rhythmic_density'], 'beat_times', 'rhythmic', 'Guitars Rhythmic Density'),
    ('rhythmic', ['voice_rhythmic_density'], 'beat_times', 'rhythmic', 'Voice Rhythmic Density'),
    ('melodic', ['voice_melodic_contour'], 'beat_times', 'rhythmic', 'Melodic Voice Contour'),
    ('melodic', ['guitars_harmonic_contour'], 'beat_times', 'rhythmic', 'Harmonic Guitars Contour'),
    ('dynamic', ['voice_loudness'], 'beat_times', 'rhythmic', 'Voice Loudness'),
    ('dynamic', ['guitars_loudness'], 'beat_times', 'rhythmic', 'Guitars Loudness'),
    ('harmonic', ['tonal_dissonance'], 'beat_times', 'rhythmic', 'Tonal Dissonance'),
    ('harmonic', ['tonal_dispersion'], 'beat_times', 'rhythmic', 'Tonal Dispersion'),
]

feature_names = { '/'.join(config[1]): config[4] for config in features_config }

results = {}
for feature_category, feature_key_path, time_key, time_category, feature_name in features_config:
    print(f"Analyzing {feature_name}")
    results['/'.join(feature_key_path)] = analyze_feature(data, feature_category, feature_key_path, time_key, time_category)

for feature_key, df_feature in results.items():
    friendly_name = feature_names.get(feature_key, feature_key)
    if method_selector == 'cosine':
        plot_feature_clusters_cosine(df_feature, friendly_name)
    elif method_selector == 'dtw':
        plot_feature_clusters_dtw(df_feature, friendly_name)

# EMD plots


In [ ]:

import numpy as np
import pandas as pd
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from PyEMD import EMD

# --- BEGIN MODIFICATION: Moved from cell 79_Yrn2ma15u ---
# Ensure `data` and `data_processed` are available globally
# Assuming `data` holds the original raw song objects and `data_processed` holds the EMD processed ones
# If these variables are not present, this will raise an error.
if 'data' not in globals() or not isinstance(data, list) or not data:
    raise NameError("Global variable 'data' (raw song data) not found or is empty.")
if 'data_processed' not in globals() or not isinstance(data_processed, list) or not data_processed:
    raise NameError("Global variable 'data_processed' (EMD data) not found or is empty.")

ALL_RAW_DATA = data
ALL_EMD_DATA = data_processed

# ============================================
# PARAM_INFO with contextual unit labels added
# ============================================
# Adjust unit_label to match your actual computation units.
PARAM_INFO = [
    # Rhythmic / timing
    {
        "display_name": "Tempo deviation",
        "raw_path": "rhythmic.tempo_deviations",
        "emd_label": "Tempo deviation",
        # Use "ΔTempo (%)" if values are relative deviations; otherwise in BPM:
        "unit_label": "ΔTempo (BPM)"
    },
    {
        "display_name": "Voice rhythmic density",
        "raw_path": "rhythmic.voice_rhythmic_density",
        "emd_label": "Voice rhythmic density",
        # If this is per unit time, consider "Onsets per second" or "Onsets per beat"
        "unit_label": "Nr. of onsets"
    },
    {
        "display_name": "Guitars rhythmic density",
        "raw_path": "rhythmic.guitars_rhythmic_density",
        "emd_label": "Guitars rhythmic density",
        "unit_label": "Nr. of onsets"
    },

    # Dynamics
    {
        "display_name": "Voice loudness",
        "raw_path": "dynamic.voice_loudness",
        "emd_label": "Voice loudness",
        "unit_label": "Loudness (dB)"
    },
    {
        "display_name": "Guitars loudness",
        "raw_path": "dynamic.guitars_loudness",
        "emd_label": "Guitars loudness",
        "unit_label": "Loudness (dB)"
    },

    # Harmonic
    {
        "display_name": "Tonal dissonance",
        "raw_path": "harmonic.tonal_dissonance",
        "emd_label": "Tonal dissonance",
        # Replace a.u. with a concrete scale if available
        "unit_label": "Dissonance (a.u.)"
    },
    {
        "display_name": "Tonal dispersion",
        "raw_path": "harmonic.tonal_dispersion",
        "emd_label": "Tonal dispersion",
        "unit_label": "Dispersion (a.u.)"
    },

    # Melodic / pitch
    {
        "display_name": "Melodic voice contour",
        "raw_path": "melodic.voice_melodic_contour",
        "emd_label": "Melodic voice contour",
        # Use "(MIDI)" if values are MIDI note numbers; "(cents)" if detailed pitch tracking
        "unit_label": "Pitch contour (semitones)"
    },
    {
        "display_name": "Harmonic guitars contour",
        "raw_path": "melodic.guitars_harmonic_contour",
        "emd_label": "Harmonic guitars contour",
        "unit_label": "Pitch contour (semitones)"
    }
]
# --- END MODIFICATION ---

# ===========================
# Helper functions (original)
# ===========================
def to_float_array(v):
    if v is None:
        return None
    try:
        if hasattr(v, "to_numpy"):
            arr = v.to_numpy(dtype=float)
        else:
            arr = np.asarray(v, dtype=float)
    except Exception:
        return None
    if arr.ndim == 0:
        arr = arr.reshape(1)
    elif arr.ndim > 1:
        arr = arr.squeeze()
        if arr.ndim > 1:
            arr = arr.ravel()
    return arr

def get_path(dct, path):
    if dct is None or not isinstance(path, str) or not path:
        return None
    cur = dct
    for part in path.split('.'):
        if isinstance(cur, dict) and part in cur:
            cur = cur.get(part)
        else:
            return None
    return cur

def _get_param_info_by_display_name(display_name):
    for p in PARAM_INFO:
        if p['display_name'] == display_name:
            return p
    return None

def _get_feature_values_raw(song_obj, display_name):
    # This function specifically gets raw data, ignoring EMD data source option
    param_info = _get_param_info_by_display_name(display_name)
    if not param_info:
        return None
    return to_float_array(get_path(song_obj, param_info['raw_path']))

def _song_label(song_obj):
    md = song_obj.get("metadata", {})
    title = md.get("title", "Untitled")
    artist = md.get("artist", "Unknown")
    year = md.get("project_year", "")
    return f"{title} — {artist}" + (f" ({year})" if str(year).strip() else "")

def _is_1d_numeric_series(x):
    if x is None:
        return False
    try:
        arr = np.asarray(x)
    except Exception:
        return False
    return arr.ndim == 1 and np.issubdtype(arr.dtype, np.number)

def _make_structural_dfs(song_obj):
    """Creates DataFrames for sections and phrases from a raw song object."""
    S = song_obj.get("structural", {})
    df_sections = None
    df_phrases = None
    st, sl = S.get("section_times"), S.get("section_labels")
    if _is_1d_numeric_series(st) and isinstance(sl, (list, tuple)) and len(st) == len(sl):
        df_sections = pd.DataFrame({"Initial_Time": st, "Label": sl})
    pt, pl = S.get("phrase_times"), S.get("phrase_labels")
    if _is_1d_numeric_series(pt) and isinstance(pl, (list, tuple)) and len(pt) == len(pl):
        df_phrases = pd.DataFrame({"Initial_Time": pt, "Label": pl})
    return df_sections, df_phrases

def format_axes_emd(ax, x_axis, df_sections, df_phrases, mode="time", beat_times=None, base_fontsize=14):
    """Bottom axis: time or beats; top axis: the other. Adds sections & phrases."""
    def sec_to_minsec(t):
        m = int(t // 60)
        s = int(round(t % 60))
        return f"{m}:{s:02d}"

    x_axis = np.asarray(x_axis)
    if x_axis.size == 0:
        return

    n_ticks = 10
    x_lo = x_axis[0] if x_axis.size > 0 else 0
    x_hi = x_axis[-1] if x_axis.size > 0 else 1
    tick_positions = np.linspace(x_lo, x_hi, n_ticks)

    bt = np.asarray(beat_times) if beat_times is not None else x_axis

    if mode == "beats":
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([f"{b+1}" for b in np.searchsorted(bt, tick_positions)], fontsize=base_fontsize)
        ax.set_xlabel("Beat", fontsize=base_fontsize)

        ax_top = ax.secondary_xaxis('top')
        ax_top.set_xticks(tick_positions)
        ax_top.set_xticklabels([sec_to_minsec(t) for t in tick_positions], fontsize=base_fontsize)
        ax_top.set_xlabel("Time (mm:ss)", fontsize=base_fontsize)
    else:
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([sec_to_minsec(t) for t in tick_positions], fontsize=base_fontsize)
        ax.set_xlabel("Time (mm:ss)", fontsize=base_fontsize)

        ax_top = ax.secondary_xaxis('top')
        ax_top.set_xticks(tick_positions)
        ax_top.set_xticklabels([f"{b+1}" for b in np.searchsorted(bt, tick_positions)], fontsize=base_fontsize)
        ax_top.set_xlabel("Beat", fontsize=base_fontsize)

    # Sections
    if df_sections is not None and all(c in df_sections.columns for c in ["Initial_Time", "Label"]):
        for time, label in zip(df_sections["Initial_Time"], df_sections["Label"]):
            ax.axvline(x=time, color="black", lw=1.2, alpha=0.9)
            ax.text(time, -0.20, label, rotation=90, ha="center", va="top",
                    fontsize=base_fontsize, color="black", transform=ax.get_xaxis_transform(), clip_on=False)

    # Phrases
    if df_phrases is not None and all(c in df_phrases.columns for c in ["Initial_Time", "Label"]):
        for t, lab in zip(df_phrases["Initial_Time"], df_phrases["Label"]):
            ax.axvline(x=t, color="lightgrey", lw=1.0, alpha=0.7)
            ax.text(t, 1.25, lab, rotation=90, ha="center", va="bottom",
                    fontsize=base_fontsize, color="grey", transform=ax.get_xaxis_transform(), clip_on=False)

    ax.tick_params(axis='y', labelsize=base_fontsize)
    ax.margins(x=0.05)
    ax.grid(False)

# ============================================
# NEW: y-axis label helpers backed by PARAM_INFO
# ============================================
def get_y_axis_label_from_param_info(display_name: str) -> str:
    """
    Returns a contextual y-axis label (with units) for a given feature display name,
    reading it from PARAM_INFO['unit_label']. Falls back to 'Value (a.u.)'.
    """
    p = _get_param_info_by_display_name(display_name)
    if p and isinstance(p, dict):
        unit_label = p.get("unit_label", None)
        if isinstance(unit_label, str) and unit_label.strip():
            return unit_label
    return "Value (a.u.)"  # Fallback if not found or empty

def format_y_axis(ax, display_name: str, base_fontsize: int = 14):
    """
    Set y-axis label and style based on the selected feature's entry in PARAM_INFO.
    """
    ylab = get_y_axis_label_from_param_info(display_name)
    ax.set_ylabel(ylab, fontsize=base_fontsize)
    ax.tick_params(axis='y', labelsize=base_fontsize)

# ============================================
# Plotting routine/callback (FIXED: y-label now applied)
# ============================================
def plot_emd_components(song_obj, display_name, base_fontsize=14):
    with out_emd:
        clear_output(wait=True)

        raw_signal = _get_feature_values_raw(song_obj, display_name)
        if raw_signal is None or len(raw_signal) == 0 or np.all(np.isnan(raw_signal)):
            print(f"No raw data found for '{display_name}' in selected song or signal is constant/all NaN.")
            return

        # Handle NaNs before EMD
        signal_for_emd = np.copy(raw_signal)
        finite_mask = np.isfinite(signal_for_emd)
        if np.any(finite_mask) and not np.all(finite_mask):
            series = pd.Series(signal_for_emd)
            signal_for_emd = series.interpolate(method='linear', limit_direction='both').values
        if np.any(np.isnan(signal_for_emd)):
            signal_for_emd = np.nan_to_num(signal_for_emd, nan=np.nanmean(signal_for_emd) if np.any(finite_mask) else 0.0)
        elif not np.any(finite_mask):
            signal_for_emd = np.zeros_like(signal_for_emd)

        if np.all(signal_for_emd == signal_for_emd[0]):
            print(f"EMD cannot be applied to a constant signal for '{display_name}'.")
            return

        emd = EMD()
        try:
            imfs = emd(signal_for_emd)
        except Exception as e:
            print(f"EMD failed for '{display_name}': {e}")
            return

        # Prepare components for plotting
        imf1 = imfs[0] if imfs.shape[0] > 0 else np.zeros_like(raw_signal)
        imf2 = imfs[1] if imfs.shape[0] > 1 else np.zeros_like(raw_signal)
        imf3_plus = np.sum(imfs[2:], axis=0) if imfs.shape[0] > 2 else np.zeros_like(raw_signal)

        # X-axis (beat times from raw song object)
        beat_times = get_path(song_obj, "rhythmic.beat_times")
        if _is_1d_numeric_series(beat_times):
            x_axis = np.asarray(beat_times, dtype=float)
            mode = "time"
        else:
            x_axis = np.arange(len(raw_signal))
            mode = "index"

        # Ensure lengths match
        min_len = min(len(x_axis), len(raw_signal))
        x_axis = x_axis[:min_len]
        raw_signal = raw_signal[:min_len]
        imf1 = imf1[:min_len]
        imf2 = imf2[:min_len]
        imf3_plus = imf3_plus[:min_len]

        # Structural info
        df_sections, df_phrases = _make_structural_dfs(song_obj)
        md = song_obj.get("metadata", {})
        plot_title = f"EMD Components for {display_name} - {md.get('title', 'Untitled')}"

        # Plot all signals together
        fig_combined, ax_combined = plt.subplots(1, 1, figsize=(12, 5), constrained_layout=True)
        ax_combined.plot(x_axis, raw_signal, label='Raw Signal', color='blue', alpha=0.7)
        ax_combined.plot(x_axis, imf1, label='IMF1', color='green', alpha=0.7)
        ax_combined.plot(x_axis, imf2, label='IMF2', color='red', alpha=0.7)
        ax_combined.plot(x_axis, imf3_plus, label='IMF3+', color='purple', alpha=0.7)

        fig_combined.suptitle(plot_title, fontsize=base_fontsize + 4, y=1.1)
        ax_combined.legend(loc='upper right', fontsize=base_fontsize)

        # Format x-axes and ADD CONTEXTUAL Y-AXIS LABEL
        format_axes_emd(ax_combined, x_axis, df_sections, df_phrases, mode=mode, beat_times=beat_times, base_fontsize=base_fontsize)
        format_y_axis(ax_combined, display_name, base_fontsize=base_fontsize)  # <-- FIX: apply y-axis label

        plt.show()

# Build song list for dropdown
song_options_emd = []
title_to_index_emd = {}
for idx, song in enumerate(ALL_RAW_DATA):  # Use ALL_RAW_DATA for consistent song indexing and metadata
    lbl = _song_label(song)
    while lbl in title_to_index_emd:
        lbl = f"{lbl} #{idx}"
    title_to_index_emd[lbl] = idx
    song_options_emd.append(lbl)

feature_options_emd = sorted([p['display_name'] for p in PARAM_INFO])

# Widgets
song_dd_emd = widgets.Dropdown(options=song_options_emd, description='Song:', layout=widgets.Layout(width='60%'))
feature_dd_emd = widgets.Dropdown(options=feature_options_emd, description='Feature:', layout=widgets.Layout(width='50%'))
out_emd = widgets.Output()

def on_change_emd(*args):
    selected_song_label = song_dd_emd.value
    selected_feature_name = feature_dd_emd.value
    if selected_song_label and selected_feature_name:
        idx = title_to_index_emd[selected_song_label]
        song_obj = ALL_RAW_DATA[idx]  # Always use raw data for EMD input
        plot_emd_components(song_obj, selected_feature_name)

# Wire up the widgets
song_dd_emd.observe(on_change_emd, names='value')
feature_dd_emd.observe(on_change_emd, names='value')

# Layout and display
display(widgets.VBox([
    song_dd_emd,
       feature_dd_emd,
]), out_emd)

# Initial plot


# Final Plots (Raw and EMD)

In [ ]:

# --- Imports ---
import numpy as np
import pandas as pd
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from scipy.ndimage import gaussian_filter1d

# -------------------------------------------------------------
# 0) DATASET DISCOVERY (Now directly uses global data and data_processed)
# -------------------------------------------------------------

# Ensure `data` and `data_processed` are available globally
# Assuming `data` holds the original raw song objects and `data_processed` holds the EMD processed ones
# If these variables are not present, this will raise an error.
if 'data' not in globals() or not isinstance(data, list) or not data:
    raise NameError("Global variable 'data' (raw song data) not found or is empty.")
if 'data_processed' not in globals() or not isinstance(data_processed, list) or not data_processed:
    raise NameError("Global variable 'data_processed' (EMD data) not found or is empty.")

ALL_RAW_DATA = data
ALL_EMD_DATA = data_processed

# -------------------------------------------------------------
# 1) CORE HELPERS
# -------------------------------------------------------------

def to_float_array(v):
    """Convert array-like to 1D float numpy array; preserve NaNs."""
    if v is None:
        return None
    try:
        if hasattr(v, "to_numpy"):
            arr = v.to_numpy(dtype=float)
        else:
            arr = np.asarray(v, dtype=float)
    except Exception:
        return None
    if arr.ndim == 0:
        arr = arr.reshape(1)
    elif arr.ndim > 1:
        arr = np.squeeze(arr)
        if arr.ndim > 1:
            arr = arr.ravel()
    return arr

def get_path(dct, path):
    """Traverse nested dict with dot-path (e.g., 'rhythmic.bpms_raw')."""
    if dct is None or not isinstance(path, str) or not path:
        return None
    cur = dct
    for part in path.split('.'):
        if isinstance(cur, dict) and part in cur:
            cur = cur.get(part)
        else:
            return None
    return cur

def format_title_from_metadata(template, song):
    """Fill template using metadata keys."""
    md = (song or {}).get("metadata", {}) if isinstance(song, dict) else {}
    return template.format(
        title=md.get("title", "Untitled"),
        artist=md.get("artist", "Unknown"),
        project_year=md.get("project_year", ""),
        sample_rate=md.get("sample_rate", "")
    )

# -------------------------------------------------------------
# Helper functions for feature access based on data source type
# -------------------------------------------------------------

# This list defines all features, their raw data paths, and EMD processed labels
PARAM_INFO = [
    {"display_name": "Tempo Deviation", "raw_path": "rhythmic.tempo_deviations", "emd_label": "Tempo deviation"},
    {"display_name": "Voice Rhythmic Density", "raw_path": "rhythmic.voice_rhythmic_density", "emd_label": "Voice rhythmic density"},
    {"display_name": "Guitars Rhythmic Density", "raw_path": "rhythmic.guitars_rhythmic_density", "emd_label": "Guitars rhythmic density"},
    {"display_name": "Voice Loudness", "raw_path": "dynamic.voice_loudness", "emd_label": "Voice loudness"},
    {"display_name": "Guitars Loudness", "raw_path": "dynamic.guitars_loudness", "emd_label": "Guitars loudness"},
    {"display_name": "Tonal Dissonance", "raw_path": "harmonic.tonal_dissonance", "emd_label": "Tonal dissonance"},
    {"display_name": "Tonal Dispersion", "raw_path": "harmonic.tonal_dispersion", "emd_label": "Tonal dispersion"},
    {"display_name": "Melodic Voice Contour", "raw_path": "melodic.voice_melodic_contour", "emd_label": "Melodic voice contour"},
    {"display_name": "Harmonic Guitars Contour", "raw_path": "melodic.guitars_harmonic_contour", "emd_label": "Harmonic guitars contour"},
]

# PARAM_MAP will now just map display_name to itself for dropdown purposes
PARAM_MAP = {p['display_name']: p['display_name'] for p in PARAM_INFO}

def _get_param_info_by_display_name(display_name):
    for p in PARAM_INFO:
        if p['display_name'] == display_name:
            return p
    return None

def _get_feature_values(song_obj, display_name, data_source_type):
    param_info = _get_param_info_by_display_name(display_name)
    if not param_info:
        return None

    if data_source_type == 'Raw Data':
        return to_float_array(get_path(song_obj, param_info['raw_path']))
    elif data_source_type == 'EMD Data':
        return to_float_array(song_obj.get('features', {}).get(param_info['emd_label'], None))
    return None

def _is_feature_available(song_obj, display_name, data_source_type):
    val = _get_feature_values(song_obj, display_name, data_source_type)
    return val is not None and val.size > 0 # Check if it's not empty array either

def _available_params_for_song(song_obj, data_source_type):
    avail = []
    for p_info in PARAM_INFO:
        if _is_feature_available(song_obj, p_info['display_name'], data_source_type):
            avail.append(p_info['display_name'])
    return sorted(avail)


# -------------------------------------------------------------
# 2) PLOTTING UTILITIES
# -------------------------------------------------------------

def minmax_normalize(x):
    x = np.asarray(x, dtype=float)
    finite = np.isfinite(x)
    if not np.any(finite):
        return np.zeros_like(x)
    lo, hi = np.nanmin(x[finite]), np.nanmax(x[finite])
    if hi - lo < 1e-9:
        y = np.zeros_like(x)
    else:
        y = (x - lo) / (hi - lo)
    y[~finite] = np.nan
    return y

def format_axes(ax, x_axis, df_sections, df_phrases, title=None, mode="time", beat_times=None):
    """Bottom axis: time or beats; top axis: the other. Adds sections & phrases."""
    def sec_to_minsec(t):
        m = int(t // 60)
        s = int(round(t % 60))
        return f"{m}:{s:02d}"

    x_axis = np.asarray(x_axis)
    if x_axis.size == 0:
        return

    n_ticks = 10
    x_lo = x_axis[0] if x_axis.size > 0 else 0
    x_hi = x_axis[-1] if x_axis.size > 0 else 1
    tick_positions = np.linspace(x_lo, x_hi, n_ticks)

    bt = np.asarray(beat_times) if beat_times is not None else x_axis

    if mode == "beats":
        ax.set_xticks(tick_positions)
        beat_indices = np.searchsorted(bt, tick_positions)
        ax.set_xticklabels([f"{b+1}" for b in beat_indices], fontsize=9)
        ax.set_xlabel("Beat")

        ax_top = ax.secondary_xaxis('top')
        ax_top.set_xticks(tick_positions)
        ax_top.set_xticklabels([sec_to_minsec(t) for t in tick_positions], fontsize=9)
        ax_top.set_xlabel("Time (mm:ss)")
    else:
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([sec_to_minsec(t) for t in tick_positions], fontsize=9)
        ax.set_xlabel("Time (mm:ss)")

        ax_top = ax.secondary_xaxis('top')
        ax_top.set_xticks(tick_positions)
        beat_indices = np.searchsorted(bt, tick_positions)
        ax_top.set_xticklabels([f"{b+1}" for b in beat_indices], fontsize=9)
        ax_top.set_xlabel("Beat")

    # Sections
    if df_sections is not None and all(c in df_sections.columns for c in ["Initial_Time", "Label"]):
        for time, label in zip(df_sections["Initial_Time"], df_sections["Label"]):
            ax.axvline(x=time, color="black", lw=1.2, alpha=0.9)
            ax.text(time, -0.18, label, rotation=90, ha="center", va="top",
                    fontsize=8, color="black", transform=ax.get_xaxis_transform(), clip_on=False)

    # Phrases
    if df_phrases is not None and all(c in df_phrases.columns for c in ["Initial_Time", "Label"]):
        for t, lab in zip(df_phrases["Initial_Time"], df_phrases["Label"]):
            ax.axvline(x=t, color="lightgrey", lw=1.0, alpha=0.7)
            ax.text(t, 1.15, lab, rotation=90, ha="center", va="bottom",
                    fontsize=8, color="grey", transform=ax.get_xaxis_transform(), clip_on=False)

    if title:
        ax.set_title(title, pad=70)

    ax.margins(x=0.05)
    ax.grid(False)

def plot(variables, norm=False, smoothing=0, x_axis=None, df_sections=None, df_phrases=None,
         title=None, mode="time", plot_index=None, peaks=None, y_label=None,
         song=None, title_template=None, beat_times=None, data_source_type='Raw Data'): # Added data_source_type
    """
    variables: list of specs; each can be
      - "Guitars Loudness" (display name)
      - ("Guitars Loudness", True, 2, "Guitars Loudness")
    song: required for feature lookup (either raw or processed song object)
    data_source_type: 'Raw Data' or 'EMD Data'
    """
    import os

    def resolve_series(display_name):
        """Return a float array and apply masks (for raw data) based on display_name."""
        y = _get_feature_values(song, display_name, data_source_type)

        if y is None:
            return None

        # Apply masks only if data_source_type is 'Raw Data'
        # For 'EMD Data', masks (NaNs) are already part of the processed features.
        if data_source_type == 'Raw Data':
            low = display_name.lower()
            # Apply masks where appropriate
            if "guitar" in low:
                m = to_float_array(get_path(song, "dynamic.g_mask"))
                if m is not None and m.size > 0:
                    L = min(len(y), len(m))
                    y = y[:L].copy()
                    m = m[:L].astype(bool)
                    y[~m] = np.nan
            if "voice" in low:
                m = to_float_array(get_path(song, "dynamic.v_mask"))
                if m is not None and m.size > 0:
                    L = min(len(y), len(m))
                    y = y[:L].copy()
                    m = m[:L].astype(bool)
                    y[~m] = np.nan
        return y

    # Final title (template uses metadata)
    final_title = title
    if final_title is None and title_template:
        final_title = format_title_from_metadata(title_template, song)
    elif final_title and "{" in final_title:
        final_title = format_title_from_metadata(final_title, song)

    if isinstance(variables, list):
        fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)

        global_min, global_max = np.inf, -np.inf
        plotted_any = False

        for var in variables:
            if isinstance(var, tuple):
                # Expecting (display_name, nrm, smooth, label)
                if len(var) == 4:
                    display_name, nrm, smooth, label = var
                elif len(var) == 3:
                    display_name, nrm, smooth = var
                    label = str(display_name)
                else:
                    display_name, nrm, smooth, label = var, norm, smoothing, str(var)
            else:
                display_name, nrm, smooth, label = var, norm, smoothing, str(var)

            y = resolve_series(display_name)
            if y is None or len(y) == 0:
                print(f"Variable '{display_name}' not found or empty for {data_source_type}.")
                continue

            y_plot = np.copy(y)
            if nrm:
                y_plot = minmax_normalize(y_plot)
            if smooth > 0:
                finite_mask = np.isfinite(y_plot)
                if np.any(finite_mask):
                    y_plot[finite_mask] = gaussian_filter1d(y_plot[finite_mask], sigma=smooth)

            # x-values
            if x_axis is not None:
                x_vals = np.asarray(x_axis)
                min_len = min(len(x_vals), len(y_plot))
                x_vals = x_vals[:min_len]
                y_plot = y_plot[:min_len]
            else:
                x_vals = np.arange(len(y_plot))

            y_masked = np.ma.masked_invalid(y_plot)
            ax.plot(x_vals, y_masked, label=label)
            plotted_any = True

            finite_y_plot = y_plot[np.isfinite(y_plot)]
            if finite_y_plot.size > 0 and not nrm:
                global_min = min(global_min, np.min(finite_y_plot))
                global_max = max(global_max, np.max(finite_y_plot))

            if peaks and display_name in peaks: # peaks should now map to display names
                peak_indices = np.asarray(peaks[display_name])
                valid_peak_indices = peak_indices[peak_indices < len(x_vals)]
                ax.scatter(x_vals[valid_peak_indices], y_plot[valid_peak_indices],
                           color='red', label=f"{label} Peaks")

        # Y-limits
        if not norm and np.isfinite(global_min) and np.isfinite(global_max):
            y_range = global_max - global_min
            if y_range < 1e-9:
                ax.set_ylim(global_min - 0.1, global_max + 0.1)
            else:
                ax.set_ylim(global_min - y_range * 0.05, global_max + y_range * 0.05)
        elif norm:
            ax.set_ylim(-0.05, 1.05)

        # Axes formatting
        if x_axis is not None and isinstance(x_axis, (list, np.ndarray)) and len(x_axis) > 0:
            format_axes(ax, x_axis, df_sections, df_phrases, title=final_title, mode=mode, beat_times=beat_times)
        else:
            if final_title:
                ax.set_title(final_title, pad=100)

        if y_label is not None:
            ax.set_ylabel(y_label)

        if plotted_any:
            ax.legend()

        if plot_index is not None:
            os.makedirs("./saved_plots", exist_ok=True)
            plot_path = os.path.join("./saved_plots", f"plot_{plot_index}.pdf")
            plt.savefig(plot_path, bbox_inches="tight")
            print(f"Saved plot to {plot_path}")

        plt.show()

# -------------------------------------------------------------
# 3) UI (Song, Var1, Var2 + Normalize/Smooth) LIMITED TO REQUESTED PARAMETERS
# -------------------------------------------------------------

def _is_1d_numeric_series(x):
    if x is None:
        return False
    try:
        arr = np.asarray(x)
    except Exception:
        return False
    return arr.ndim == 1 and np.issubdtype(arr.dtype, np.number)

def _make_structural_dfs(song_obj):
    """Creates DataFrames for sections and phrases from a raw song object."""
    S = song_obj.get("structural", {})
    df_sections = None
    df_phrases = None
    st, sl = S.get("section_times"), S.get("section_labels")
    if _is_1d_numeric_series(st) and isinstance(sl, (list, tuple)) and len(st) == len(sl):
        df_sections = pd.DataFrame({"Initial_Time": st, "Label": sl})
    pt, pl = S.get("phrase_times"), S.get("phrase_labels")
    if _is_1d_numeric_series(pt) and isinstance(pl, (list, tuple)) and len(pt) == len(pl):
        df_phrases = pd.DataFrame({"Initial_Time": pt, "Label": pl})
    return df_sections, df_phrases

def _song_label(song_obj):
    md = song_obj.get("metadata", {})
    title = md.get("title", "Untitled")
    artist = md.get("artist", "Unknown")
    year = md.get("project_year", "")
    return f"{title} — {artist}" + (f" ({year})" if str(year).strip() else "")

# Build song list for dropdown (always based on raw data for consistent indexing/labeling)
song_options = []
title_to_index = {}
for idx, song in enumerate(ALL_RAW_DATA):
    lbl = _song_label(song)
    while lbl in title_to_index:
        lbl = f"{lbl}  #{idx}"
    title_to_index[lbl] = idx
    song_options.append(lbl)

# Widgets
data_source_dd = widgets.Dropdown(options=['Raw Data', 'EMD Data'], description='Data Source:')
song_dd = widgets.Dropdown(options=song_options, description='Song:', layout=widgets.Layout(width='60%'))

var1_dd  = widgets.Dropdown(options=[], description='Var 1:', layout=widgets.Layout(width='50%'))
norm1_cb = widgets.Checkbox(value=False, description='Normalize 1')
smooth1_sl = widgets.IntSlider(value=0, min=0, max=6, step=1, description='Smooth 1σ', layout=widgets.Layout(width='35%'))

var2_dd  = widgets.Dropdown(options=[], description='Var 2:', layout=widgets.Layout(width='50%'))
norm2_cb = widgets.Checkbox(value=False, description='Normalize 2')
smooth2_sl = widgets.IntSlider(value=0, min=0, max=6, step=1, description='Smooth 2σ', layout=widgets.Layout(width='35%'))

out = widgets.Output()

# State variables
current_song = None             # The song object currently selected for plotting (raw or processed)
current_raw_song = None         # The corresponding raw song object (for structural info)
beat_times_for_axes = None
AVAILABLE_PARAMS = []           # list of display names available in current song/data source

def _rebuild_song_state(selected_label, data_source_type):
    """Load the selected song and compute x-axis, structures, and available parameters."""
    global current_song, current_raw_song, beat_times_for_axes, AVAILABLE_PARAMS

    # 1. Determine which list of songs to use for features
    if data_source_type == 'Raw Data':
        current_features_list = ALL_RAW_DATA
    else: # 'EMD Data'
        current_features_list = ALL_EMD_DATA

    # 2. Get the index from the selected label (which refers to ALL_RAW_DATA's index)
    idx = title_to_index[selected_label]

    # 3. Set the current song object for feature plotting
    current_song = current_features_list[idx]

    # 4. Always get the *raw* song object for structural info (sections, phrases, beat_times)
    current_raw_song = ALL_RAW_DATA[idx]

    # 5. Determine available parameters for the selected song and data source type
    AVAILABLE_PARAMS = _available_params_for_song(current_song, data_source_type)

    # 6. Extract x-axis data and structural markers from the raw song object
    bt = get_path(current_raw_song, "rhythmic.beat_times")
    if _is_1d_numeric_series(bt):
        beat_times_for_axes = np.asarray(bt, dtype=float)
        x_axis = beat_times_for_axes
        mode = "time"
    else:
        pt = get_path(current_raw_song, "structural.phrase_times")
        if _is_1d_numeric_series(pt):
            x_axis = np.asarray(pt, dtype=float)
            beat_times_for_axes = None
            mode = "time"
        else:
            # If no beat_times or phrase_times, use index over the longest available param series
            longest = 0
            for p_info in PARAM_INFO:
                s = _get_feature_values(current_song, p_info['display_name'], data_source_type)
                if s is not None:
                    longest = max(longest, len(s))
            x_axis = np.arange(longest) if longest > 0 else np.array([])
            beat_times_for_axes = None
            mode = "time"

    df_sections, df_phrases = _make_structural_dfs(current_raw_song)
    plot_title_template = "{title} — {artist} ({project_year})\n" + f"Data Source: {data_source_type}"

    return AVAILABLE_PARAMS, x_axis, df_sections, df_phrases, plot_title_template, mode

def _refresh_var_dropdowns():
    keys = AVAILABLE_PARAMS
    var1_dd.options = keys
    var2_dd.options = keys
    # sensible defaults
    prefs = [
        "Tempo Deviation",
        "Voice Loudness",
        "Guitars Loudness",
        "Voice Rhythmic Density",
        "Guitars Rhythmic Density",
        "Tonal Dissonance",
        "Tonal Dispersion",
        "Melodic Voice Contour",
        "Harmonic Guitars Contour",
    ]
    var1_dd.value = next((k for k in prefs if k in keys), (keys[0] if keys else None))
    var2_dd.value = next((k for k in prefs if k in keys and k != var1_dd.value),
                         (keys[1] if len(keys) > 1 else None))

def _draw_plot(*_):
    out.clear_output(wait=True)
    if current_song is None or current_raw_song is None:
        with out:
            print("Please select a song.")
        return

    data_source_type = data_source_dd.value # Get data source type from widget
    params_local, x_axis, df_sections, df_phrases, plot_title_template, mode = \
        _rebuild_song_state(song_dd.value, data_source_type)

    variables = []
    # Build specs using selected display names and carry normalization/smoothing
    if var1_dd.value is not None and var1_dd.value in params_local:
        variables.append((var1_dd.value, norm1_cb.value, smooth1_sl.value, var1_dd.value))
    if var2_dd.value is not None and var2_dd.value in params_local:
        variables.append((var2_dd.value, norm2_cb.value, smooth2_sl.value, var2_dd.value))

    with out:
        plot(
            variables=variables,
            x_axis=x_axis,
            df_sections=df_sections,
            df_phrases=df_phrases,
            title=None, # title template is set inside rebuild_song_state
            title_template=plot_title_template,
            mode=mode,
            y_label=None,
            plot_index=None,
            peaks=None,
            song=current_song, # current_song is either raw or processed, for feature data
            beat_times=beat_times_for_axes,
            data_source_type=data_source_type # Pass data source type to plot function
        )

def _on_song_change(change):
    if change['name'] == 'value' and change['new'] is not None:
        _rebuild_song_state(change['new'], data_source_dd.value)
        _refresh_var_dropdowns()
        _draw_plot()

def _on_data_source_change(change):
    if change['name'] == 'value' and change['new'] is not None:
        _rebuild_song_state(song_dd.value, change['new'])
        _refresh_var_dropdowns()
        _draw_plot()

# Initialize & wire
if song_options:
    # Initial state rebuild with default selections
    _rebuild_song_state(song_options[0], data_source_dd.value)
    _refresh_var_dropdowns()

song_dd.observe(_on_song_change, names='value')
data_source_dd.observe(_on_data_source_change, names='value') # New observer for data source
var1_dd.observe(lambda ch: _draw_plot(), names='value')
var2_dd.observe(lambda ch: _draw_plot(), names='value')
norm1_cb.observe(lambda ch: _draw_plot(), names='value')
norm2_cb.observe(lambda ch: _draw_plot(), names='value')
smooth1_sl.observe(lambda ch: _draw_plot(), names='value')
smooth2_sl.observe(lambda ch: _draw_plot(), names='value')

# Layout & display
controls = widgets.VBox([
    data_source_dd, # Add data source dropdown to the layout
    song_dd,
    widgets.HBox([var1_dd, norm1_cb, smooth1_sl]),
    widgets.HBox([var2_dd, norm2_cb, smooth2_sl]),
])
display(controls, out)

# Initial render
_draw_plot()

In [ ]:

# --- Imports ---
import numpy as np
import pandas as pd
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from scipy.ndimage import gaussian_filter1d

# Ensure all plot text is at least 14 pt
plt.rcParams.update({
    'font.size': 14,            # default text
    'axes.titlesize': 14,       # axes title
    'axes.labelsize': 14,       # x/y labels
    'xtick.labelsize': 14,      # x tick labels
    'ytick.labelsize': 14,      # y tick labels
    'legend.fontsize': 14       # legend text
})

# ------------------------------------------------------------
# 0) DATASET DISCOVERY (Now directly uses global data and data_processed)
# ------------------------------------------------------------
# Ensure `data` and `data_processed` are available globally
# Assuming `data` holds the original raw song objects and `data_processed` holds the EMD processed ones
# If these variables are not present, this will raise an error.
if 'data' not in globals() or not isinstance(data, list) or not data:
    raise NameError("Global variable 'data' (raw song data) not found or is empty.")
if 'data_processed' not in globals() or not isinstance(data_processed, list) or not data_processed:
    raise NameError("Global variable 'data_processed' (EMD data) not found or is empty.")

ALL_RAW_DATA = data
ALL_EMD_DATA = data_processed

# ------------------------------------------------------------
# 1) CORE HELPERS
# ------------------------------------------------------------
def to_float_array(v):
    """Convert array-like to 1D float numpy array; preserve NaNs."""
    if v is None:
        return None
    try:
        if hasattr(v, "to_numpy"):
            arr = v.to_numpy(dtype=float)
        else:
            arr = np.asarray(v, dtype=float)
    except Exception:
        return None
    if arr.ndim == 0:
        arr = arr.reshape(1)
    elif arr.ndim > 1:
        arr = np.squeeze(arr)
    if arr.ndim > 1:
        arr = arr.ravel()
    return arr

def get_path(dct, path):
    """Traverse nested dict with dot-path (e.g., 'rhythmic.bpms_raw')."""
    if dct is None or not isinstance(path, str) or not path:
        return None
    cur = dct
    for part in path.split('.'):
        if isinstance(cur, dict) and part in cur:
            cur = cur.get(part)
        else:
            return None
    return cur

def format_title_from_metadata(template, song):
    """Fill template using metadata keys."""
    md = (song or {}).get("metadata", {}) if isinstance(song, dict) else {}
    return template.format(
        title=md.get("title", "Untitled"),
        artist=md.get("artist", "Unknown"),
        project_year=md.get("project_year", ""),
        sample_rate=md.get("sample_rate", "")
    )

# ------------------------------------------------------------
# Helper functions for feature access based on data source type
# ------------------------------------------------------------
PARAM_INFO = [
    {"display_name": "Tempo Deviation", "raw_path": "rhythmic.tempo_deviations", "emd_label": "Tempo deviation"},
    {"display_name": "Voice Rhythmic Density", "raw_path": "rhythmic.voice_rhythmic_density", "emd_label": "Voice rhythmic density"},
    {"display_name": "Guitars Rhythmic Density", "raw_path": "rhythmic.guitars_rhythmic_density", "emd_label": "Guitars rhythmic density"},
    {"display_name": "Voice Loudness", "raw_path": "dynamic.voice_loudness", "emd_label": "Voice loudness"},
    {"display_name": "Guitars Loudness", "raw_path": "dynamic.guitars_loudness", "emd_label": "Guitars loudness"},
    {"display_name": "Tonal Dissonance", "raw_path": "harmonic.tonal_dissonance", "emd_label": "Tonal dissonance"},
    {"display_name": "Tonal Dispersion", "raw_path": "harmonic.tonal_dispersion", "emd_label": "Tonal dispersion"},
    {"display_name": "Melodic Voice Contour", "raw_path": "melodic.voice_melodic_contour", "emd_label": "Melodic voice contour"},
    {"display_name": "Harmonic Guitars Contour", "raw_path": "melodic.guitars_harmonic_contour", "emd_label": "Harmonic guitars contour"},
]
PARAM_MAP = {p['display_name']: p['display_name'] for p in PARAM_INFO}

def _get_param_info_by_display_name(display_name):
    for p in PARAM_INFO:
        if p['display_name'] == display_name:
            return p
    return None

def _get_feature_values(song_obj, display_name, data_source_type):
    param_info = _get_param_info_by_display_name(display_name)
    if not param_info:
        return None
    if data_source_type == 'Raw Data':
        return to_float_array(get_path(song_obj, param_info['raw_path']))
    elif data_source_type == 'EMD Data':
        return to_float_array(song_obj.get('features', {}).get(param_info['emd_label'], None))
    return None

def _is_feature_available(song_obj, display_name, data_source_type):
    val = _get_feature_values(song_obj, display_name, data_source_type)
    return val is not None and val.size > 0

def _available_params_for_song(song_obj, data_source_type):
    avail = []
    for p_info in PARAM_INFO:
        if _is_feature_available(song_obj, p_info['display_name'], data_source_type):
            avail.append(p_info['display_name'])
    return sorted(avail)

# ------------------------------------------------------------
# 2) PLOTTING UTILITIES
# ------------------------------------------------------------
def minmax_normalize(x):
    x = np.asarray(x, dtype=float)
    finite = np.isfinite(x)
    if not np.any(finite):
        return np.zeros_like(x)
    lo, hi = np.nanmin(x[finite]), np.nanmax(x[finite])
    if hi - lo < 1e-9:
        y = np.zeros_like(x)
    else:
        y = (x - lo) / (hi - lo)
    y[~finite] = np.nan
    return y

def format_axes(ax, x_axis, df_sections, df_phrases, title=None, mode="time", beat_times=None):
    """Bottom axis: time or beats; top axis: the other. Adds sections & phrases."""
    def sec_to_minsec(t):
        m = int(t // 60)
        s = int(round(t % 60))
        return f"{m}:{s:02d}"

    x_axis = np.asarray(x_axis)
    if x_axis.size == 0:
        return

    n_ticks = 10
    x_lo = x_axis[0] if x_axis.size > 0 else 0
    x_hi = x_axis[-1] if x_axis.size > 0 else 1
    tick_positions = np.linspace(x_lo, x_hi, n_ticks)

    bt = np.asarray(beat_times) if beat_times is not None else x_axis

    if mode == "beats":
        ax.set_xticks(tick_positions)
        beat_indices = np.searchsorted(bt, tick_positions)
        ax.set_xticklabels([f"{b+1}" for b in beat_indices], fontsize=14)
        ax.set_xlabel("Beat", fontsize=14)

        ax_top = ax.secondary_xaxis('top')
        ax_top.set_xticks(tick_positions)
        ax_top.set_xticklabels([sec_to_minsec(t) for t in tick_positions], fontsize=14)
        ax_top.set_xlabel("Time (mm:ss)", fontsize=14)
    else:
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([sec_to_minsec(t) for t in tick_positions], fontsize=14)
        ax.set_xlabel("Time (mm:ss)", fontsize=14)

        ax_top = ax.secondary_xaxis('top')
        ax_top.set_xticks(tick_positions)
        beat_indices = np.searchsorted(bt, tick_positions)
        ax_top.set_xticklabels([f"{b+1}" for b in beat_indices], fontsize=14)
        ax_top.set_xlabel("Beat", fontsize=14)

    # Sections
    if df_sections is not None and all(c in df_sections.columns for c in ["Initial_Time", "Label"]):
        for time, label in zip(df_sections["Initial_Time"], df_sections["Label"]):
            ax.axvline(x=time, color="black", lw=1.2, alpha=0.9)
            ax.text(
                time, -0.18, label, rotation=90, ha="center", va="top",
                fontsize=14, color="black", transform=ax.get_xaxis_transform(), clip_on=False
            )

    # Phrases
    if df_phrases is not None and all(c in df_phrases.columns for c in ["Initial_Time", "Label"]):
        for t, lab in zip(df_phrases["Initial_Time"], df_phrases["Label"]):
            ax.axvline(x=t, color="lightgrey", lw=1.0, alpha=0.7)
            ax.text(
                t, 1.15, lab, rotation=90, ha="center", va="bottom",
                fontsize=14, color="grey", transform=ax.get_xaxis_transform(), clip_on=False
            )

    if title:
        ax.set_title(title, pad=70, fontsize=14)
    ax.margins(x=0.05)
    ax.grid(False)

def plot(
    variables, norm=False, smoothing=0, x_axis=None, df_sections=None, df_phrases=None,
    title=None, mode="time", plot_index=None, peaks=None, y_label=None,
    song=None, title_template=None, beat_times=None, data_source_type='Raw Data'
):
    """
    variables: list of specs; each can be
      - "Guitars Loudness" (display name)
      - ("Guitars Loudness", True, 2, "Guitars Loudness")
    song: required for feature lookup (either raw or processed song object)
    data_source_type: 'Raw Data' or 'EMD Data'
    """
    import os

    def resolve_series(display_name):
        """Return a float array and apply masks (for raw data) based on display_name."""
        y = _get_feature_values(song, display_name, data_source_type)
        if y is None:
            return None

        # Apply masks only if data_source_type is 'Raw Data'
        if data_source_type == 'Raw Data':
            low = display_name.lower()
            if "guitar" in low:
                m = to_float_array(get_path(song, "dynamic.g_mask"))
                if m is not None and m.size > 0:
                    L = min(len(y), len(m))
                    y = y[:L].copy()
                    m = m[:L].astype(bool)
                    y[~m] = np.nan
            if "voice" in low:
                m = to_float_array(get_path(song, "dynamic.v_mask"))
                if m is not None and m.size > 0:
                    L = min(len(y), len(m))
                    y = y[:L].copy()
                    m = m[:L].astype(bool)
                    y[~m] = np.nan

        return y

    # Final title (template uses metadata)
    final_title = title
    if final_title is None and title_template:
        final_title = format_title_from_metadata(title_template, song)
    elif final_title and "{" in final_title:
        final_title = format_title_from_metadata(final_title, song)

    if isinstance(variables, list):
        fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)

        global_min, global_max = np.inf, -np.inf
        plotted_any = False

        for var in variables:
            if isinstance(var, tuple):
                # Expecting (display_name, nrm, smooth, label)
                if len(var) == 4:
                    display_name, nrm, smooth, label = var
                elif len(var) == 3:
                    display_name, nrm, smooth = var
                    label = str(display_name)
                else:
                    display_name, nrm, smooth, label = var, norm, smoothing, str(var)
            else:
                display_name, nrm, smooth, label = var, norm, smoothing, str(var)

            y = resolve_series(display_name)
            if y is None or len(y) == 0:
                print(f"Variable '{display_name}' not found or empty for {data_source_type}.")
                continue

            y_plot = np.copy(y)

            if nrm:
                y_plot = minmax_normalize(y_plot)

            if smooth > 0:
                finite_mask = np.isfinite(y_plot)
                if np.any(finite_mask):
                    y_plot[finite_mask] = gaussian_filter1d(y_plot[finite_mask], sigma=smooth)

            # x-values
            if x_axis is not None:
                x_vals = np.asarray(x_axis)
                min_len = min(len(x_vals), len(y_plot))
                x_vals = x_vals[:min_len]
                y_plot = y_plot[:min_len]
            else:
                x_vals = np.arange(len(y_plot))

            y_masked = np.ma.masked_invalid(y_plot)
            ax.plot(x_vals, y_masked, label=label)
            plotted_any = True

            finite_y_plot = y_plot[np.isfinite(y_plot)]
            if finite_y_plot.size > 0 and not nrm:
                global_min = min(global_min, np.min(finite_y_plot))
                global_max = max(global_max, np.max(finite_y_plot))

            if peaks and display_name in peaks:  # peaks should now map to display names
                peak_indices = np.asarray(peaks[display_name])
                valid_peak_indices = peak_indices[peak_indices < len(x_vals)]
                ax.scatter(
                    x_vals[valid_peak_indices], y_plot[valid_peak_indices],
                    color='red', label=f"{label} Peaks"
                )

        # Y-limits
        if not norm and np.isfinite(global_min) and np.isfinite(global_max):
            y_range = global_max - global_min
            if y_range < 1e-9:
                ax.set_ylim(global_min - 0.1, global_max + 0.1)
            else:
                ax.set_ylim(global_min - y_range * 0.05, global_max + y_range * 0.05)
        elif norm:
            ax.set_ylim(-0.05, 1.05)

        # Axes formatting
        if x_axis is not None and isinstance(x_axis, (list, np.ndarray)) and len(x_axis) > 0:
            format_axes(ax, x_axis, df_sections, df_phrases, title=final_title, mode=mode, beat_times=beat_times)
        else:
            if final_title:
                ax.set_title(final_title, pad=100, fontsize=14)

        # Y-label
        if y_label is not None:
            ax.set_ylabel(y_label, fontsize=14)

        if plotted_any:
            ax.legend(fontsize=14)

        if plot_index is not None:
            os.makedirs("./saved_plots", exist_ok=True)
            plot_path = os.path.join("./saved_plots", f"plot_{plot_index}.pdf")
            plt.savefig(plot_path, bbox_inches="tight")
            print(f"Saved plot to {plot_path}")

        plt.show()

# ------------------------------------------------------------
# 3) UI (Song, Var1, Var2 + Normalize/Smooth) LIMITED TO REQUESTED PARAMETERS
# ------------------------------------------------------------
def _is_1d_numeric_series(x):
    if x is None:
        return False
    try:
        arr = np.asarray(x)
    except Exception:
        return False
    return arr.ndim == 1 and np.issubdtype(arr.dtype, np.number)

def _make_structural_dfs(song_obj):
    """Creates DataFrames for sections and phrases from a raw song object."""
    S = song_obj.get("structural", {})
    df_sections = None
    df_phrases = None

    st, sl = S.get("section_times"), S.get("section_labels")
    if _is_1d_numeric_series(st) and isinstance(sl, (list, tuple)) and len(st) == len(sl):
        df_sections = pd.DataFrame({"Initial_Time": st, "Label": sl})

    pt, pl = S.get("phrase_times"), S.get("phrase_labels")
    if _is_1d_numeric_series(pt) and isinstance(pl, (list, tuple)) and len(pt) == len(pl):
        df_phrases = pd.DataFrame({"Initial_Time": pt, "Label": pl})

    return df_sections, df_phrases

def _song_label(song_obj):
    md = song_obj.get("metadata", {})
    title = md.get("title", "Untitled")
    artist = md.get("artist", "Unknown")
    year = md.get("project_year", "")
    return f"{title} — {artist}" + (f" ({year})" if str(year).strip() else "")

# Build song list for dropdown (always based on raw data for consistent indexing/labeling)
song_options = []
title_to_index = {}
for idx, song in enumerate(ALL_RAW_DATA):
    lbl = _song_label(song)
    while lbl in title_to_index:
        lbl = f"{lbl} #{idx}"
    title_to_index[lbl] = idx
    song_options.append(lbl)

# Widgets
data_source_dd = widgets.Dropdown(options=['Raw Data', 'EMD Data'], description='Data Source:')
song_dd = widgets.Dropdown(options=song_options, description='Song:', layout=widgets.Layout(width='60%'))
var1_dd = widgets.Dropdown(options=[], description='Var 1:', layout=widgets.Layout(width='50%'))
norm1_cb = widgets.Checkbox(value=False, description='Normalize 1')
smooth1_sl = widgets.IntSlider(value=0, min=0, max=6, step=1, description='Smooth 1σ', layout=widgets.Layout(width='35%'))

var2_dd = widgets.Dropdown(options=[], description='Var 2:', layout=widgets.Layout(width='50%'))
norm2_cb = widgets.Checkbox(value=False, description='Normalize 2')
smooth2_sl = widgets.IntSlider(value=0, min=0, max=6, step=1, description='Smooth 2σ', layout=widgets.Layout(width='35%'))

out = widgets.Output()

# State variables
current_song = None       # The song object currently selected for plotting (raw or processed)
current_raw_song = None   # The corresponding raw song object (for structural info)
beat_times_for_axes = None
AVAILABLE_PARAMS = []     # list of display names available in current song/data source

def _rebuild_song_state(selected_label, data_source_type):
    """Load the selected song and compute x-axis, structures, and available parameters."""
    global current_song, current_raw_song, beat_times_for_axes, AVAILABLE_PARAMS

    # 1. Determine which list of songs to use for features
    if data_source_type == 'Raw Data':
        current_features_list = ALL_RAW_DATA
    else:  # 'EMD Data'
        current_features_list = ALL_EMD_DATA

    # 2. Get the index from the selected label (which refers to ALL_RAW_DATA's index)
    idx = title_to_index[selected_label]

    # 3. Set the current song object for feature plotting
    current_song = current_features_list[idx]

    # 4. Always get the *raw* song object for structural info (sections, phrases, beat_times)
    current_raw_song = ALL_RAW_DATA[idx]

    # 5. Determine available parameters for the selected song and data source type
    AVAILABLE_PARAMS = _available_params_for_song(current_song, data_source_type)

    # 6. Extract x-axis data and structural markers from the raw song object
    bt = get_path(current_raw_song, "rhythmic.beat_times")
    if _is_1d_numeric_series(bt):
        beat_times_for_axes = np.asarray(bt, dtype=float)
        x_axis = beat_times_for_axes
        mode = "time"
    else:
        pt = get_path(current_raw_song, "structural.phrase_times")
        if _is_1d_numeric_series(pt):
            x_axis = np.asarray(pt, dtype=float)
            beat_times_for_axes = None
            mode = "time"
        else:
            # If no beat_times or phrase_times, use index over the longest available param series
            longest = 0
            for p_info in PARAM_INFO:
                s = _get_feature_values(current_song, p_info['display_name'], data_source_type)
                if s is not None:
                    longest = max(longest, len(s))
            x_axis = np.arange(longest) if longest > 0 else np.array([])
            beat_times_for_axes = None
            mode = "time"

    df_sections, df_phrases = _make_structural_dfs(current_raw_song)
    plot_title_template = "{title} — {artist} ({project_year})\n" + f"Data Source: {data_source_type}"
    return AVAILABLE_PARAMS, x_axis, df_sections, df_phrases, plot_title_template, mode

def _refresh_var_dropdowns():
    keys = AVAILABLE_PARAMS
    var1_dd.options = keys
    var2_dd.options = keys

    # sensible defaults
    prefs = [
        "Tempo Deviation",
        "Voice Loudness",
        "Guitars Loudness",
        "Voice Rhythmic Density",
        "Guitars Rhythmic Density",
        "Tonal Dissonance",
        "Tonal Dispersion",
        "Melodic Voice Contour",
        "Harmonic Guitars Contour",
    ]
    var1_dd.value = next((k for k in prefs if k in keys), (keys[0] if keys else None))
    var2_dd.value = next((k for k in prefs if k in keys and k != var1_dd.value),
                         (keys[1] if len(keys) > 1 else None))

def _draw_plot(*_):
    out.clear_output(wait=True)
    if current_song is None or current_raw_song is None:
        with out:
            print("Please select a song.")
        return

    data_source_type = data_source_dd.value  # Get data source type from widget
    params_local, x_axis, df_sections, df_phrases, plot_title_template, mode = \
        _rebuild_song_state(song_dd.value, data_source_type)

    variables = []

    # Build specs using selected display names and carry normalization/smoothing
    active_vars = []  # list of (name, normalized_bool)
    if var1_dd.value is not None and var1_dd.value in params_local:
        variables.append((var1_dd.value, norm1_cb.value, smooth1_sl.value, var1_dd.value))
        active_vars.append((var1_dd.value, norm1_cb.value))
    if var2_dd.value is not None and var2_dd.value in params_local:
        variables.append((var2_dd.value, norm2_cb.value, smooth2_sl.value, var2_dd.value))
        active_vars.append((var2_dd.value, norm2_cb.value))

    # ---------- y-axis label logic ----------
    def compose_y_label(pairs):
        """
        pairs: list of (display_name, is_normalized)
        Rules:
          - If both normalized -> "Normalized values"
          - Else -> show corresponding metric(s), adding "(normalized)" to any normalized one
        """
        if len(pairs) == 0:
            return None

        all_norm = all(is_norm for _, is_norm in pairs)
        if all_norm:
            return "Normalized values"

        if len(pairs) == 1:
            name, is_norm = pairs[0]
            return f"{name}" + (" (normalized)" if is_norm else "")

        # len == 2
        (name1, norm1), (name2, norm2) = pairs[0], pairs[1]
        if name1 == name2:
            if norm1 and norm2:
                return "Normalized values"
            elif norm1 or norm2:
                return f"{name1} (normalized)"
            else:
                return name1
        else:
            if norm1 and norm2:
                return "Normalized values"
            elif norm1 and not norm2:
                return f"{name1} (normalized) & {name2}"
            elif norm2 and not norm1:
                return f"{name1} & {name2} (normalized)"
            else:
                return f"{name1} / {name2}"

    y_label_str = compose_y_label(active_vars)
    # Global normalization flag for y-limits: True only if both variables are normalized
    global_norm_flag = (len(active_vars) > 0) and all(is_norm for _, is_norm in active_vars)
    # ---------- end y-axis label logic ----------

    with out:
        plot(
            variables=variables,
            x_axis=x_axis,
            df_sections=df_sections,
            df_phrases=df_phrases,
            title=None,  # title template is set inside rebuild_song_state
            title_template=plot_title_template,
            mode=mode,
            y_label=y_label_str,          # pass the computed y-axis label
            plot_index=None,
            peaks=None,
            song=current_song,            # current_song is either raw or processed, for feature data
            beat_times=beat_times_for_axes,
            data_source_type=data_source_type,
            norm=global_norm_flag          # ensure y-limits match normalization state
        )

def _on_song_change(change):
    if change['name'] == 'value' and change['new'] is not None:
        _rebuild_song_state(change['new'], data_source_dd.value)
        _refresh_var_dropdowns()
        _draw_plot()

def _on_data_source_change(change):
    if change['name'] == 'value' and change['new'] is not None:
        _rebuild_song_state(song_dd.value, change['new'])
        _refresh_var_dropdowns()
        _draw_plot()

# Initialize & wire
if song_options:
    _rebuild_song_state(song_options[0], data_source_dd.value)
    _refresh_var_dropdowns()

# --- Correct observers ---
song_dd.observe(_on_song_change, names='value')
data_source_dd.observe(_on_data_source_change, names='value')
var1_dd.observe(lambda ch: _draw_plot(), names='value')
var2_dd.observe(lambda ch: _draw_plot(), names='value')
norm1_cb.observe(lambda ch: _draw_plot(), names='value')
norm2_cb.observe(lambda ch: _draw_plot(), names='value')
smooth1_sl.observe(lambda ch: _draw_plot(), names='value')
smooth2_sl.observe(lambda ch: _draw_plot(), names='value')

# Layout & display
controls = widgets.VBox([
    data_source_dd,
    song_dd,
    widgets.HBox([var1_dd, norm1_cb, smooth1_sl]),
    widgets.HBox([var2_dd, norm2_cb, smooth2_sl]),
])
display(controls, out)

# Initial render


# MDS Viz

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import MDS
from sklearn.metrics import pairwise_distances

# --- 1. Prepare all correlation matrices for MDS ---
# Re-generate piece-level correlation matrices with associated metadata

piece_correlation_matrices_with_metadata = []

def flatten_upper_triangle(matrix):
    # Extract upper triangle (excluding diagonal) to get unique correlations
    mask = np.triu(np.ones(matrix.shape), k=1).astype(bool)
    return matrix.values[mask]

for processed_row in data_processed:
    title = processed_row.get('metadata', {}).get('title', 'Unknown Title')
    artist = processed_row.get('metadata', {}).get('artist', 'Unknown Artist')
    features = processed_row.get('features', {})

    # Filter out masks and keep only numeric features
    piece_data = {k: v for k, v in features.items()
                  if isinstance(v, list) and k not in ['Voice Mask', 'Guitar Mask'] and len(v) > 0}

    # Ensure all features have equal length and enough data for correlation
    if len(piece_data) > 1:
        lengths = {len(vals) for vals in piece_data.values()}
        if len(lengths) == 1 and list(lengths)[0] > 1: # Ensure at least 2 data points for correlation
            df_piece = pd.DataFrame(piece_data)
            correlation_matrix = df_piece.corr(method='pearson')

            piece_correlation_matrices_with_metadata.append({
                'title': title,
                'artist': artist,
                'matrix': correlation_matrix
            })

# Filter to include only piece-level data for MDS
flattened_vectors = []
labels = [] # Will be track titles
artists_for_plot = [] # Will be artists for coloring

for entry in piece_correlation_matrices_with_metadata:
    flat_vec = flatten_upper_triangle(entry['matrix'])
    flattened_vectors.append(flat_vec)
    labels.append(entry['title'])
    artists_for_plot.append(entry['artist'])

# Convert to numpy array
X = np.array(flattened_vectors)

# --- 2. Compute pairwise distances ---
# Using Euclidean distance between the flattened vectors
distance_matrix = pairwise_distances(X, metric='euclidean')

# --- 3. Apply MDS ---
# n_components=2 for a 2D plot
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42)
X_transformed = mds.fit_transform(distance_matrix)

# Create a DataFrame for easy plotting
df_mds = pd.DataFrame(X_transformed, columns=['MDS1', 'MDS2'])
df_mds['Label'] = labels # Track title
df_mds['Artist'] = artists_for_plot # Artist for coloring

# --- 4. Visualize the results ---
plt.figure(figsize=(12, 10))
sns.scatterplot(data=df_mds, x='MDS1', y='MDS2', hue='Artist', s=150, alpha=0.8)

# Annotate points with their labels (track names)
for i, row in df_mds.iterrows():
    plt.annotate(row['Label'], (row['MDS1'] + 0.02, row['MDS2'] + 0.02), fontsize=9)

plt.title('MDS of Pearson Correlation Matrices (Individual Pieces)')
plt.xlabel('MDS Dimension 1')
plt.ylabel('MDS Dimension 2')
plt.grid(True)
plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.axvline(0, color='grey', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

# Bootstrap analysis

We evaluated whether the cutoff |r| ≥ 0.6 was appropriate by estimating 95% bootstrap confidence intervals (moving‑block bootstrap, B=1000) for every pairwise correlation across all pieces. No feature pair in the corpus exhibited a confidence interval entirely above 0.6, whereas a substantial majority had confidence intervals entirely below this threshold. The remaining near‑threshold cases displayed wide confidence intervals crossing 0.6 and highly variable bootstrap probabilities of exceeding the threshold, indicating that these correlations are not statistically stable. Thus, 0.6 acts as a conservative but empirically justified boundary: correlations below it are reliably weak, while correlations above it would require evidence stronger than what is present in the data.



In [ ]:

# ============================================
# Bootstrap Pearson correlations — TABLES ONLY
# ============================================
import os
import numpy as np
import pandas as pd
import itertools
from math import ceil
from IPython.display import display

# ------------------------------
# CONFIG (adjust as needed)
# ------------------------------
BOOTSTRAP_B    = 1000          # number of bootstrap replicates
CI             = (2.5, 97.5)   # percentile CI (95%)
THRESHOLD      = 0.6           # correlation threshold to test
ABSOLUTE       = True          # use |r| (True) or signed r (False)
RANDOM_STATE   = 42            # reproducible results

# Show & save tables
SHOW_TABLES    = True          # display tables in notebook
SAVE_TABLES    = False         # save CSVs to disk
TABLE_DIR      = "./bootstrap_tables"

# Optional diagnostics
ADD_EFFECTIVE_N = True         # include n_eff (pairwise non-NaN count) per pair

# Name of the probability column (used consistently for sorting/saving)
PROB_COL = f"p({'|r|≥' if ABSOLUTE else 'r≥'}{THRESHOLD:g})"

# ------------------------------
# Bootstrap helpers
# ------------------------------
def _moving_block_indices(n, block_length, rng):
    """
    Build bootstrap indices of length n by concatenating randomly chosen
    contiguous blocks of length block_length, then trim to n.
    If block_length is None or <=1, performs i.i.d. bootstrap.
    """
    if block_length is None or block_length <= 1:
        return rng.integers(0, n, size=n)
    Bn = ceil(n / block_length)
    starts = rng.integers(0, max(1, n - block_length + 1), size=Bn)
    idx = []
    for s in starts:
        idx.extend(range(s, min(s + block_length, n)))
    return np.array(idx[:n])

def bootstrap_corr_matrix(
    df_piece,
    B=BOOTSTRAP_B,
    ci=CI,
    threshold=THRESHOLD,
    absolute=ABSOLUTE,
    block_length=None,     # e.g., int(round(n**(1/3))) for time series
    random_state=RANDOM_STATE
):
    """
    Bootstrap Pearson correlations for a piece (columns = features, rows = time points).
    Returns median, CI_low, CI_high, and prob(|r| >= threshold) matrices with same shape.
    Uses pandas pairwise complete observations (NaNs allowed).
    """
    rng = np.random.default_rng(random_state)
    cols = df_piece.columns
    n = len(df_piece)

    boot_mats = []
    for _ in range(B):
        idx = _moving_block_indices(n, block_length, rng)
        sample_df = df_piece.iloc[idx]
        C = sample_df.corr(method='pearson')  # pairwise complete obs
        boot_mats.append(C.values)

    boot_arr = np.stack(boot_mats, axis=0)  # shape: (B, p, p)

    # summarize over B
    boot_arr_use = np.abs(boot_arr) if absolute else boot_arr
    med = np.nanmedian(boot_arr_use, axis=0)
    lo  = np.nanpercentile(boot_arr_use, ci[0], axis=0)
    hi  = np.nanpercentile(boot_arr_use, ci[1], axis=0)

    # probability of exceeding threshold
    if absolute:
        prob_above = np.nanmean(np.abs(boot_arr) >= threshold, axis=0)
    else:
        prob_above = np.nanmean(boot_arr >= threshold, axis=0)

    med_df = pd.DataFrame(med, index=cols, columns=cols)
    lo_df  = pd.DataFrame(lo,  index=cols, columns=cols)
    hi_df  = pd.DataFrame(hi,  index=cols, columns=cols)
    p_df   = pd.DataFrame(prob_above, index=cols, columns=cols)

    return med_df, lo_df, hi_df, p_df

def _default_block_length(n):
    """Rule-of-thumb for short-range dependent series (min 5)."""
    return max(5, int(round(n ** (1/3))))

def _upper_triangle_pairs(labels):
    """Yield (i, j, label_i, label_j) for upper triangle only (exclude diagonal)."""
    p = len(labels)
    for i in range(p):
        for j in range(i+1, p):
            yield i, j, labels[i], labels[j]

# ------------------------------
# Run per-piece bootstrap and produce TABLES
# ------------------------------
# Expects 'data_processed' to exist (EMD/masked features with 'metadata' and 'features')
data_sorted_by_artist = sorted(
    data_processed,
    key=lambda x: x.get('metadata', {}).get('artist', 'Unknown Artist')
)
artist_groups = itertools.groupby(
    data_sorted_by_artist,
    key=lambda x: x.get('metadata', {}).get('artist', 'Unknown Artist')
)

if SAVE_TABLES and not os.path.exists(TABLE_DIR):
    os.makedirs(TABLE_DIR, exist_ok=True)

piece_bootstrap_summaries = []   # per-track summary rows

for artist, pieces in artist_groups:
    artist_pieces = list(pieces)

    for row in artist_pieces:
        title = row.get('metadata', {}).get('title', 'Unknown')
        features = row.get('features', {})

        # keep numeric lists only; drop mask keys
        piece_data = {
            k: v for k, v in features.items()
            if isinstance(v, list) and k not in ['Voice Mask', 'Guitar Mask'] and len(v) > 0
        }

        # ensure equal length across features
        if len(piece_data) <= 1:
            print(f"⚠️ Skipping {title}: not enough features.")
            continue
        lengths = {len(vals) for vals in piece_data.values()}
        if len(lengths) != 1:
            print(f"⚠️ Skipping {title}: features have different lengths.")
            continue

        df_piece = pd.DataFrame(piece_data)
        n = len(df_piece)
        bl = _default_block_length(n)  # change to None for i.i.d. bootstrap

        # --- Bootstrap ---
        med_df, lo_df, hi_df, p_df = bootstrap_corr_matrix(
            df_piece,
            B=BOOTSTRAP_B,
            ci=CI,
            threshold=THRESHOLD,
            absolute=ABSOLUTE,
            block_length=bl,
            random_state=RANDOM_STATE
        )

        # --- Flags relative to the threshold ---
        stable_strong = (lo_df >= THRESHOLD)   # CI entirely above threshold
        stable_weak   = (hi_df <  THRESHOLD)   # CI entirely below threshold
        uncertain     = ~(stable_strong | stable_weak)

        # --- Build tables (upper triangle only) ---
        rows_borderline = []
        rows_strong     = []
        rows_weak       = []

        labels = list(med_df.columns)

        # Optional: precompute n_eff for all pairs from the ORIGINAL df_piece (not bootstrapped)
        n_eff_cache = {}
        if ADD_EFFECTIVE_N:
            for i, j, li, lj in _upper_triangle_pairs(labels):
                n_eff_cache[(li, lj)] = int(df_piece[[li, lj]].dropna().shape[0])

        for i, j, li, lj in _upper_triangle_pairs(labels):
            row_common = {
                "Artist": artist,
                "Title": title,
                "Var_i": li,
                "Var_j": lj,
                "median": float(med_df.iat[i, j]),
                "CI_low": float(lo_df.iat[i, j]),
                "CI_high": float(hi_df.iat[i, j]),
                PROB_COL: float(p_df.iat[i, j])
            }
            if ADD_EFFECTIVE_N:
                row_common["n_eff"] = n_eff_cache.get((li, lj), np.nan)

            if uncertain.iat[i, j]:
                rows_borderline.append(row_common)
            elif stable_strong.iat[i, j]:
                rows_strong.append(row_common)
            elif stable_weak.iat[i, j]:
                rows_weak.append(row_common)

        # Create DataFrames
        df_borderline = pd.DataFrame(rows_borderline)
        df_strong     = pd.DataFrame(rows_strong)
        df_weak       = pd.DataFrame(rows_weak)

        # Sort ONLY if non-empty
        if not df_borderline.empty:
            sort_col = PROB_COL if PROB_COL in df_borderline.columns else ("median" if "median" in df_borderline.columns else None)
            if sort_col:
                df_borderline = df_borderline.sort_values(sort_col, ascending=False)

        if not df_strong.empty:
            sort_col = PROB_COL if PROB_COL in df_strong.columns else ("median" if "median" in df_strong.columns else None)
            if sort_col:
                df_strong = df_strong.sort_values(sort_col, ascending=False)

        if not df_weak.empty:
            # for weak pairs it's often informative to sort by median ascending
            sort_col = "median" if "median" in df_weak.columns else (PROB_COL if PROB_COL in df_weak.columns else None)
            if sort_col:
                df_weak = df_weak.sort_values(sort_col, ascending=True)

        # --- Show tables (notebook) ---
        if SHOW_TABLES:
            print(f"\n=== {title} — {artist} (n={n}, block={bl}) ===")
            def show_table(label, df, none_msg="none"):
                print(f"• {label}:")
                if df.empty:
                    print(f"  {none_msg}")
                else:
                    display(df)

            show_table("Borderline pairs (CI crosses threshold)", df_borderline)
            show_table("Stable strong pairs (CI entirely ≥ threshold)", df_strong)
            show_table("Stable weak pairs (CI entirely < threshold)", df_weak)

        # --- Save tables (CSV) ---
        if SAVE_TABLES:
            safe_title = f"{artist}__{title}".replace(" ", "_").replace("/", "_")
            if not df_borderline.empty:
                df_borderline.to_csv(os.path.join(TABLE_DIR, f"borderline__{safe_title}.csv"), index=False)
            if not df_strong.empty:
                df_strong.to_csv(os.path.join(TABLE_DIR, f"strong__{safe_title}.csv"), index=False)
            if not df_weak.empty:
                df_weak.to_csv(os.path.join(TABLE_DIR, f"weak__{safe_title}.csv"), index=False)

        # --- Per-piece summary (counts over upper triangle) ---
        p = len(labels)
        n_pairs = (p * (p - 1)) // 2
        n_strong = len(rows_strong)
        n_weak   = len(rows_weak)
        n_unc    = len(rows_borderline)

        piece_bootstrap_summaries.append({
            "Artist": artist,
            "Title": title,
            "n_points": n,
            "block_len": bl,
            "pairs_total": n_pairs,
            "pairs_stable_strong": n_strong,
            "pairs_stable_weak": n_weak,
            "pairs_borderline": n_unc,
            "share_strong": n_strong / n_pairs if n_pairs else np.nan,
            "share_weak": n_weak / n_pairs if n_pairs else np.nan,
            "share_borderline": n_unc / n_pairs if n_pairs else np.nan,
        })

# ------------------------------
# Show overall per-piece summary
# ------------------------------
df_summary = pd.DataFrame(piece_bootstrap_summaries).sort_values("share_strong", ascending=False)
print("\n=== Per-piece summary ===\n")


In [ ]:

import numpy as np
import pandas as pd
from math import ceil
from IPython.display import display

# ------------------------------
# Block-index generators
# ------------------------------
def moving_block_indices(n, block_length, rng):
    """MBB: fixed-length overlapping blocks (no wrap)."""
    if block_length is None or block_length <= 1:
        return rng.integers(0, n, size=n)
    Bn = ceil(n / block_length)
    starts = rng.integers(0, max(1, n - block_length + 1), size=Bn)
    idx = []
    for s in starts:
        idx.extend(range(s, min(s + block_length, n)))
    return np.array(idx[:n])

def stationary_block_indices(n, avg_block_length, rng):
    """
    SBB: random-length blocks; geometric( p=1/avg_block_length ) for restarts,
    wrap-around indexing; conditional stationarity of resampled series.
    Politis & Romano (1994).  (avg_block_length >= 2 recommended)
    """
    if avg_block_length is None or avg_block_length <= 1:
        # fall back to iid bootstrap
        return rng.integers(0, n, size=n)
    p = 1.0 / float(avg_block_length)  # restart prob
    idx = np.empty(n, dtype=int)
    # start at a random position
    pos = rng.integers(0, n)
    idx[0] = pos
    for t in range(1, n):
        # restart with prob p; otherwise continue block
        if rng.random() < p:
            pos = rng.integers(0, n)
        else:
            pos = (pos + 1) % n  # wrap-around
        idx[t] = pos
    return idx

# ------------------------------
# Unified bootstrap for correlation matrices (MBB or SBB)
# ------------------------------
def bootstrap_corr_matrix(
    df_piece,
    B=1000,
    ci=(2.5, 97.5),
    threshold=0.6,
    absolute=True,
    method="sbb",          # 'sbb' or 'mbb'
    block_length=None,     # for MBB: fixed l; for SBB: avg l
    random_state=42
):
    rng = np.random.default_rng(random_state)
    cols = df_piece.columns
    n = len(df_piece)

    boot_mats = []
    for _ in range(B):
        if method == "sbb":
            idx = stationary_block_indices(n, block_length, rng)
        elif method == "mbb":
            idx = moving_block_indices(n, block_length, rng)
        else:
            raise ValueError("method must be 'sbb' or 'mbb'")
        sample_df = df_piece.iloc[idx]
        C = sample_df.corr(method='pearson')  # pairwise complete obs
        boot_mats.append(C.values)

    boot_arr = np.stack(boot_mats, axis=0)  # (B, p, p)
    use = np.abs(boot_arr) if absolute else boot_arr

    med = np.nanmedian(use, axis=0)
    lo  = np.nanpercentile(use, ci[0], axis=0)
    hi  = np.nanpercentile(use, ci[1], axis=0)
    prob = (np.nanmean((np.abs(boot_arr) if absolute else boot_arr) >= threshold, axis=0))

    med_df = pd.DataFrame(med, index=cols, columns=cols)
    lo_df  = pd.DataFrame(lo,  index=cols, columns=cols)
    hi_df  = pd.DataFrame(hi,  index=cols, columns=cols)
    p_df   = pd.DataFrame(prob, index=cols, columns=cols)
    return med_df, lo_df, hi_df, p_df

def default_block_length(n):
    """Rule-of-thumb: l ~ n^(1/3) (min 5)."""
    return max(5, int(round(n ** (1/3))))

# ------------------------------
# Piece-level comparison: SBB vs MBB
# ------------------------------
def compare_bootstraps_for_piece(
    df_piece,
    B=1000,
    threshold=0.6,
    absolute=True,
    l_sbb=None,
    l_mbb=None,
    random_state=42
):
    n = len(df_piece)
    l_sbb = default_block_length(n) if l_sbb is None else l_sbb
    l_mbb = default_block_length(n) if l_mbb is None else l_mbb

    # Run SBB
    s_med, s_lo, s_hi, s_p = bootstrap_corr_matrix(
        df_piece, B=B, threshold=threshold, absolute=absolute,
        method="sbb", block_length=l_sbb, random_state=random_state
    )
    # Run MBB
    m_med, m_lo, m_hi, m_p = bootstrap_corr_matrix(
        df_piece, B=B, threshold=threshold, absolute=absolute,
        method="mbb", block_length=l_mbb, random_state=random_state
    )

    # Upper triangle only
    p = len(df_piece.columns)
    mask = np.triu(np.ones((p, p), dtype=bool), k=1)

    # Metrics
    s_width = (s_hi.values - s_lo.values)[mask]
    m_width = (m_hi.values - m_lo.values)[mask]
    width_ratio = np.nanmean(s_width / m_width)   # >1: SBB wider (more conservative)

    # Classification counts
    s_strong = int((s_lo.values[mask] >= threshold).sum())
    s_weak   = int((s_hi.values[mask] <  threshold).sum())
    s_unc    = int(mask.sum() - s_strong - s_weak)

    m_strong = int((m_lo.values[mask] >= threshold).sum())
    m_weak   = int((m_hi.values[mask] <  threshold).sum())
    m_unc    = int(mask.sum() - m_strong - m_weak)

    # Probability differences
    mean_prob_diff = float(np.nanmean(s_p.values[mask] - m_p.values[mask]))

    # Correlation of medians (agreement)
    from scipy.stats import pearsonr
    r_med = np.nan
    try:
        r_med = float(pearsonr(s_med.values[mask], m_med.values[mask])[0])
    except Exception:
        pass

    summary = pd.DataFrame({
        "method": ["SBB", "MBB"],
        "block_length": [l_sbb, l_mbb],
        "pairs_total": [int(mask.sum()), int(mask.sum())],
        "pairs_stable_strong": [s_strong, m_strong],
        "pairs_stable_weak":   [s_weak,   m_weak],
        "pairs_borderline":    [s_unc,    m_unc]
    })

    details = {
        "width_ratio_SBB_over_MBB": width_ratio,
        "mean_prob_diff_SBB_minus_MBB": mean_prob_diff,
        "median_agreement_r": r_med,
        "SBB": {"med": s_med, "lo": s_lo, "hi": s_hi, "p": s_p},
        "MBB": {"med": m_med, "lo": m_lo, "hi": m_hi, "p": m_p},
    }

    return summary, details

# ------------------------------
# Example usage inside your loop per piece:
summary, details = compare_bootstraps_for_piece(df_piece, B=1000, threshold=0.6, absolute=True)
display(summary)
print("Width ratio SBB/MBB:", details["width_ratio_SBB_over_MBB"])
print("Mean Δprob (SBB−MBB):", details["mean_prob_diff_SBB_minus_MBB"])
print("Median agreement r:", details["median_agreement_r"])


Classification stability across resampling methods was high. Only a small number of feature pairs changed classification between Stationary Block Bootstrap (SBB) and Moving‑Block Bootstrap (MBB), and all flips were between weak and borderline; none became strong under either method. MBB produced slightly more borderline classifications (11 vs 8), indicating a modest tendency to be more cautious near the threshold. This pattern does not affect the main conclusion that |r| = 0.6 is a conservative cutoff: no pair achieved a bootstrap CI entirely above 0.6 under either method.



In [ ]:

# ============================================
# SBB vs MBB comparison for Pearson correlations — TABLES ONLY
# ============================================
import os
import numpy as np
import pandas as pd
import itertools
from math import ceil
from IPython.display import display

# Try to import pearsonr; fall back to np.corrcoef if SciPy isn't present.
try:
    from scipy.stats import pearsonr
    def _corr_1d(a, b):
        try:
            return float(pearsonr(a, b)[0])
        except Exception:
            c = np.corrcoef(a, b)
            return float(c[0, 1])
except Exception:
    def _corr_1d(a, b):
        c = np.corrcoef(a, b)
        return float(c[0, 1])

# ------------------------------
# CONFIG (adjust as needed)
# ------------------------------
BOOTSTRAP_B    = 1000          # bootstrap replicates per method (reduce to 200 for quick tests)
CI             = (2.5, 97.5)   # percentile CI
THRESHOLD      = 0.6           # correlation threshold to test (on |r| if ABSOLUTE=True)
ABSOLUTE       = True          # use |r| (True) or signed r (False)
RANDOM_STATE   = 42            # reproducible resampling

# Block length rule: if None, uses n^(1/3) (min 5)
FIXED_L_SBB    = None          # set to an int to force the same avg block length in SBB (e.g., 6 or 7)
FIXED_L_MBB    = None          # set to an int to force the same fixed block length in MBB (e.g., 6 or 7)

# What to show / save
SHOW_SUMMARY_TABLES     = True   # per-piece summary
SHOW_FLIPS_TABLE        = True   # show pairs whose classification differs between SBB and MBB
SHOW_PER_PAIR_DETAILS   = False  # big table: SBB vs MBB per pair (upper triangle)
SAVE_TABLES             = False  # save CSVs
OUT_DIR                 = "./bootstrap_cmp_tables"

# ------------------------------
# Basic guards
# ------------------------------
if 'data_processed' not in globals() or not isinstance(data_processed, list) or len(data_processed) == 0:
    raise RuntimeError("This block requires 'data_processed' (list of processed song dicts) to be defined.")

# ------------------------------
# Block-index generators
# ------------------------------
def moving_block_indices(n, block_length, rng):
    """MBB: fixed-length overlapping blocks (no wrap)."""
    if block_length is None or block_length <= 1:
        return rng.integers(0, n, size=n)
    Bn = ceil(n / block_length)
    starts = rng.integers(0, max(1, n - block_length + 1), size=Bn)
    idx = []
    for s in starts:
        idx.extend(range(s, min(s + block_length, n)))
    return np.array(idx[:n])

def stationary_block_indices(n, avg_block_length, rng):
    """
    SBB: Politis & Romano (1994): random-length blocks via geometric restarts,
    wrap-around indexing; conditional stationarity of the resampled series.
    """
    if avg_block_length is None or avg_block_length <= 1:
        return rng.integers(0, n, size=n)
    p = 1.0 / float(avg_block_length)  # restart probability
    idx = np.empty(n, dtype=int)
    pos = rng.integers(0, n)  # start position
    idx[0] = pos
    for t in range(1, n):
        if rng.random() < p:
            pos = rng.integers(0, n)  # restart at random position
        else:
            pos = (pos + 1) % n       # continue block (wrap)
        idx[t] = pos
    return idx

# ------------------------------
# Unified bootstrap for correlation matrices (MBB or SBB)
# ------------------------------
def bootstrap_corr_matrix(
    df_piece,
    B=BOOTSTRAP_B,
    ci=CI,
    threshold=THRESHOLD,
    absolute=ABSOLUTE,
    method="sbb",          # 'sbb' or 'mbb'
    block_length=None,     # for SBB: average; for MBB: fixed
    random_state=RANDOM_STATE
):
    rng = np.random.default_rng(random_state)
    cols = df_piece.columns
    n = len(df_piece)

    boot_mats = []
    for _ in range(B):
        if method == "sbb":
            idx = stationary_block_indices(n, block_length, rng)
        elif method == "mbb":
            idx = moving_block_indices(n, block_length, rng)
        else:
            raise ValueError("method must be 'sbb' or 'mbb'")
        sample_df = df_piece.iloc[idx]
        C = sample_df.corr(method='pearson')  # pairwise complete obs (NaNs ok)
        boot_mats.append(C.values)

    boot_arr = np.stack(boot_mats, axis=0)  # (B, p, p)
    use = np.abs(boot_arr) if absolute else boot_arr

    med = np.nanmedian(use, axis=0)
    lo  = np.nanpercentile(use, ci[0], axis=0)
    hi  = np.nanpercentile(use, ci[1], axis=0)
    prob = np.nanmean((np.abs(boot_arr) if absolute else boot_arr) >= threshold, axis=0)

    med_df = pd.DataFrame(med, index=cols, columns=cols)
    lo_df  = pd.DataFrame(lo,  index=cols, columns=cols)
    hi_df  = pd.DataFrame(hi,  index=cols, columns=cols)
    p_df   = pd.DataFrame(prob, index=cols, columns=cols)
    return med_df, lo_df, hi_df, p_df

def default_block_length(n):
    """Rule-of-thumb: l ~ n^(1/3) (min 5)."""
    return max(5, int(round(n ** (1/3))))

def classify_matrix(lo_df, hi_df, threshold):
    """
    Return a DataFrame of labels: 'strong' if CI_low>=thr, 'weak' if CI_high<thr, else 'borderline'.
    """
    # Initialize DataFrame with object dtype to store strings
    labels = pd.DataFrame(index=lo_df.index, columns=lo_df.columns, dtype=object)
    strong = lo_df >= threshold
    weak   = hi_df <  threshold
    labels[weak]   = "weak"
    labels[strong] = "strong"
    labels[~(strong | weak)] = "borderline"
    return labels

def upper_triangle_mask(p):
    """Upper triangle (excluding diagonal)."""
    return np.triu(np.ones((p, p), dtype=bool), k=1)

# ------------------------------
# Piece-level comparison: SBB vs MBB
# ------------------------------
def compare_bootstraps_for_piece(
    df_piece, title, artist,
    B=BOOTSTRAP_B, threshold=THRESHOLD, absolute=ABSOLUTE,
    l_sbb=None, l_mbb=None, random_state=RANDOM_STATE,
    show_flips=SHOW_FLIPS_TABLE, show_details=SHOW_PER_PAIR_DETAILS,
    save_tables=SAVE_TABLES, out_dir=OUT_DIR
):
    n = len(df_piece)
    l_sbb = default_block_length(n) if l_sbb is None else l_sbb
    l_mbb = default_block_length(n) if l_mbb is None else l_mbb

    # --- SBB ---
    s_med, s_lo, s_hi, s_p = bootstrap_corr_matrix(
        df_piece, B=B, threshold=threshold, absolute=absolute,
        method="sbb", block_length=l_sbb, random_state=random_state
    )
    s_cls = classify_matrix(s_lo, s_hi, threshold)

    # --- MBB ---
    m_med, m_lo, m_hi, m_p = bootstrap_corr_matrix(
        df_piece, B=B, threshold=threshold, absolute=absolute,
        method="mbb", block_length=l_mbb, random_state=random_state
    )
    m_cls = classify_matrix(m_lo, m_hi, threshold)

    # --- Upper triangle only ---
    p = len(df_piece.columns)
    mask = upper_triangle_mask(p)
    lab = list(df_piece.columns)

    # Counts
    def count_labels(labels_df):
        vals = labels_df.values[mask]
        return (int(np.sum(vals == "strong")),
                int(np.sum(vals == "weak")),
                int(np.sum(vals == "borderline")),
                int(vals.size))

    s_strong, s_weak, s_unc, s_total = count_labels(s_cls)
    m_strong, m_weak, m_unc, m_total = count_labels(m_cls)

    # CI width & agreement
    s_width = (s_hi.values - s_lo.values)[mask]
    m_width = (m_hi.values - m_lo.values)[mask]
    with np.errstate(invalid='ignore', divide='ignore'):
        width_ratio = float(np.nanmean(s_width / (m_width + 1e-12))) if m_width.size else np.nan

    med_agree_r = np.nan
    try:
        med_agree_r = _corr_1d(s_med.values[mask], m_med.values[mask])
    except Exception:
        med_agree_r = np.nan

    mean_prob_diff = float(np.nanmean(s_p.values[mask] - m_p.values[mask])) if m_width.size else np.nan

    # Summary table (per-piece)
    df_summary = pd.DataFrame({
        "method": ["SBB", "MBB"],
        "artist": artist,
        "title": title,
        "block_length": [l_sbb, l_mbb],
        "pairs_total": [s_total, m_total],
        "pairs_stable_strong": [s_strong, m_strong],
        "pairs_stable_weak":   [s_weak,   m_weak],
        "pairs_borderline":    [s_unc,    m_unc],
        "share_strong": [s_strong/s_total if s_total else np.nan,
                         m_strong/m_total if m_total else np.nan],
        "share_weak":   [s_weak/s_total if s_total else np.nan,
                         m_weak/m_total if m_total else np.nan],
        "share_borderline":[s_unc/s_total if s_total else np.nan,
                            m_unc/m_total if m_total else np.nan],
        # piece‑level comparison metrics (same on both rows for convenience)
        "width_ratio_SBB_over_MBB": [width_ratio, width_ratio],
        "median_agreement_r":       [med_agree_r, med_agree_r],
        "mean_prob_diff_SBB_minus_MBB": [mean_prob_diff, mean_prob_diff],
    })

    if SHOW_SUMMARY_TABLES:
        print(f"\n=== {title} — {artist} (n={n}) ===")
        display(df_summary)

    # Build flips table (class differs)
    rows_flips, rows_details = [], []
    for i in range(p):
        for j in range(i+1, p):
            li, lj = lab[i], lab[j]
            cls_s = s_cls.iat[i, j]
            cls_m = m_cls.iat[i, j]
            # Always include in details when show_details=True; include in flips only if classes differ
            if show_details or cls_s != cls_m:
                s_ci_w = float(s_hi.iat[i, j] - s_lo.iat[i, j])
                m_ci_w = float(m_hi.iat[i, j] - m_lo.iat[i, j])
                row = {
                    "Artist": artist, "Title": title,
                    "Var_i": li, "Var_j": lj,
                    "SBB_class": cls_s, "MBB_class": cls_m,
                    "SBB_median": float(s_med.iat[i, j]),
                    "SBB_CI_low": float(s_lo.iat[i, j]),
                    "SBB_CI_high": float(s_hi.iat[i, j]),
                    "SBB_prob": float(s_p.iat[i, j]),
                    "MBB_median": float(m_med.iat[i, j]),
                    "MBB_CI_low": float(m_lo.iat[i, j]),
                    "MBB_CI_high": float(m_hi.iat[i, j]),
                    "MBB_prob": float(m_p.iat[i, j]),
                    "CI_width_ratio_SBB_over_MBB": float(s_ci_w / (m_ci_w + 1e-12)),
                    "delta_prob_SBB_minus_MBB": float(s_p.iat[i, j] - m_p.iat[i, j])
                }
                if cls_s != cls_m:
                    rows_flips.append(row)
                if show_details:
                    rows_details.append(row)

    df_flips = pd.DataFrame(rows_flips)
    df_details = pd.DataFrame(rows_details)

    # Sort & show
    if SHOW_FLIPS_TABLE and not df_flips.empty:
        # rank flips by closeness to threshold (SBB median proximity) and abs delta_prob
        df_flips["proximity"] = np.abs(df_flips["SBB_median"] - THRESHOLD)
        df_flips = df_flips.sort_values(["proximity", "delta_prob_SBB_minus_MBB"], ascending=[True, False])
        print("• Pairs with classification flip (SBB vs MBB):")
        display(df_flips.drop(columns=["proximity"]))

    if SHOW_PER_PAIR_DETAILS and not df_details.empty:
        # show the top 25 lines (to avoid flooding), sorted by |delta_prob|
        df_details = df_details.sort_values("delta_prob_SBB_minus_MBB", ascending=False).head(25)
        print("• Per‑pair details (top 25 by Δ probability SBB−MBB):")
        display(df_details)

    # Save CSVs if requested
    if SAVE_TABLES:
        os.makedirs(out_dir, exist_ok=True)
        safe_title = f"{artist}__{title}".replace(" ", "_").replace("/", "_")
        df_summary.to_csv(os.path.join(out_dir, f"summary__{safe_title}.csv"), index=False)
        if not df_flips.empty:
            df_flips.drop(columns=["proximity"], errors="ignore").to_csv(
                os.path.join(out_dir, f"flips__{safe_title}.csv"), index=False
            )
        if not df_details.empty:
            df_details.to_csv(os.path.join(out_dir, f"details__{safe_title}.csv"), index=False)

    # Return data for corpus-level aggregation
    return df_summary, df_flips, df_details

# ------------------------------
# Drive the comparison across your corpus (data_processed must exist)
# ------------------------------
if SAVE_TABLES and not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR, exist_ok=True)

all_piece_summaries = []
all_flips = []

# Group by artist (same approach you use elsewhere)
data_sorted_by_artist = sorted(
    data_processed, key=lambda x: x.get('metadata', {}).get('artist', 'Unknown Artist')
)
artist_groups = itertools.groupby(
    data_sorted_by_artist, key=lambda x: x.get('metadata', {}).get('artist', 'Unknown Artist')
)

for artist, pieces in artist_groups:
    for row in pieces:
        title = row.get('metadata', {}).get('title', 'Unknown')
        features = row.get('features', {})

        # keep numeric lists only; drop mask keys
        piece_data = {
            k: v for k, v in features.items()
            if isinstance(v, list) and k not in ['Voice Mask', 'Guitar Mask'] and len(v) > 0
        }

        # ensure equal length across features
        if len(piece_data) <= 1:
            print(f"⚠️ Skipping {title}: not enough features.")
            continue
        lengths = {len(vals) for vals in piece_data.values()}
        if len(lengths) != 1:
            print(f"⚠️ Skipping {title}: features have different lengths.")
            continue

        df_piece = pd.DataFrame(piece_data)
        # choose block lengths
        l_sbb = FIXED_L_SBB if FIXED_L_SBB is not None else default_block_length(len(df_piece))
        l_mbb = FIXED_L_MBB if FIXED_L_MBB is not None else default_block_length(len(df_piece))

        # run comparison
        df_sum, df_flip, _ = compare_bootstraps_for_piece(
            df_piece, title, artist,
            B=BOOTSTRAP_B, threshold=THRESHOLD, absolute=ABSOLUTE,
            l_sbb=l_sbb, l_mbb=l_mbb, random_state=RANDOM_STATE,
            show_flips=SHOW_FLIPS_TABLE, show_details=SHOW_PER_PAIR_DETAILS,
            save_tables=SAVE_TABLES, out_dir=OUT_DIR
        )
        all_piece_summaries.append(df_sum)
        if not df_flip.empty:
            all_flips.append(df_flip.assign(Artist=artist, Title=title))

# ------------------------------
# Corpus‑level summary
# ------------------------------
if all_piece_summaries:
    df_all = pd.concat(all_piece_summaries, ignore_index=True)

    # One line per method per piece; aggregate by method
    corpus_summary = (df_all
        .groupby("method", as_index=False)
        .agg({
            "pairs_total":"sum",
            "pairs_stable_strong":"sum",
            "pairs_stable_weak":"sum",
            "pairs_borderline":"sum",
            "width_ratio_SBB_over_MBB":"mean",
            "median_agreement_r":"mean",
            "mean_prob_diff_SBB_minus_MBB":"mean"
        })
    )
    # shares at the corpus level
    corpus_summary["share_strong"] = corpus_summary["pairs_stable_strong"] / corpus_summary["pairs_total"]
    corpus_summary["share_weak"]   = corpus_summary["pairs_stable_weak"]   / corpus_summary["pairs_total"]
    corpus_summary["share_borderline"] = corpus_summary["pairs_borderline"] / corpus_summary["pairs_total"]

    print("\n=== Corpus‑level summary (aggregated over pieces) ===\n")
    display(corpus_summary)

    if SAVE_TABLES:
        corpus_summary.to_csv(os.path.join(OUT_DIR, "corpus_summary.csv"), index=False)

if all_flips:
    df_flips_all = pd.concat(all_flips, ignore_index=True)
    print("\n=== All flips across the corpus (first 50) ===\n")
    display(df_flips_all.head(50))
    if SAVE_TABLES:

        # Ensure output directory exists
        os.makedirs(OUT_DIR, exist_ok=True)

        # Save the full flips table
        flips_path = os.path.join(OUT_DIR, "corpus_flips.csv")
        df_flips_all.to_csv(flips_path, index=False)
        print(f"Saved corpus flips to: {flips_path}")

        # Summarize flip types (SBB_class → MBB_class)
        df_flip_summary = (
            df_flips_all
            .assign(flip_type=df_flips_all["SBB_class"].astype(str) + " → " + df_flips_all["MBB_class"].astype(str))
            .groupby("flip_type", as_index=False)
            .size()
            .sort_values("size", ascending=False)
        )
        print("\n=== Flip type summary (SBB_class → MBB_class) ===\n")
        display(df_flip_summary)

        # Save flip summary
        flip_summary_path = os.path.join(OUT_DIR, "corpus_flip_summary.csv")
        df_flip_summary.to_csv(flip_summary_path, index=False)
        print(f"Saved flip summary to: {flip_summary_path}")

        # Cross-tab: counts of SBB_class vs MBB_class
        df_cross = (
            df_flips_all
            .pivot_table(index="SBB_class", columns="MBB_class", values="Var_i",
                         aggfunc="count", fill_value=0)
            .astype(int)
            .sort_index(axis=0)
            .sort_index(axis=1)
        )
        print("\n=== Flip cross‑tab (counts): SBB_class × MBB_class ===\n")
        display(df_cross)

        # Save cross-tab
        cross_path = os.path.join(OUT_DIR, "corpus_flip_crosstab.csv")
        df_cross.to_csv(cross_path)
        print(f"Saved flip cross‑tab to: {cross_path}")

    else:
        # If not saving, still show a compact summary in the notebook
        df_flip_summary = (
            df_flips_all
            .assign(flip_type=df_flips_all["SBB_class"].astype(str) + " → " + df_flips_all["MBB_class"].astype(str))
            .groupby("flip_type", as_index=False)
            .size()
            .sort_values("size", ascending=False)
        )
        print("\n=== Flip type summary (SBB_class → MBB_class) ===\n")
        display(df_flip_summary)

        df_cross = (
            df_flips_all
            .pivot_table(index="SBB_class", columns="MBB_class", values="Var_i",
                         aggfunc="count", fill_value=0)
            .astype(int)
            .sort_index(axis=0)
            .sort_index(axis=1)
        )
        print("\n=== Flip cross‑tab (counts): SBB_class × MBB_class ===\n")
        display(df_cross)
else:
    print("\n=== No classification flips across the corpus ===\n")


## Visualization
Generate a scatter plot of feature pair correlations across all pieces, displaying the median correlation, 95% bootstrap confidence intervals as horizontal error bars, and color-coding points based on whether the confidence interval is entirely above 0.6 (stable strong), entirely below 0.6 (stable weak), or crosses 0.6 (borderline). Include a vertical line at the 0.6 correlation threshold.

## Prepare Data for Visualization

### Subtask:
Combine bootstrap results (median, CI_low, CI_high, and classification) from all pieces into a single DataFrame suitable for plotting.


**Reasoning**:
The subtask requires aggregating bootstrap results from all pieces into a single DataFrame. This involves iterating through each processed piece, calculating its bootstrap correlation matrix, classifying each feature pair's correlation, and then compiling these details into a unified list of dictionaries which will finally be converted into a pandas DataFrame. This step will set up the data for further visualization or analysis of the bootstrap results.



In [ ]:
import numpy as np
import pandas as pd
import itertools

# Helper to classify correlation based on CI
def classify_correlation(ci_low, ci_high, threshold):
    if ci_low >= threshold:
        return "stable strong"
    elif ci_high < threshold:
        return "stable weak"
    else:
        return "borderline"

# Reusing existing bootstrap functions defined in previous cells
# _default_block_length, bootstrap_corr_matrix, and upper_triangle_pairs
# (from cell lv0RBADe1XAE and GTcY0gj-9yaS) are assumed to be defined.
# If not, they would need to be re-copied here or the previous cells executed.

all_correlations_data = []

# Iterate through each processed row (piece)
for processed_row in data_processed:
    title = processed_row.get('metadata', {}).get('title', 'Unknown Title')
    artist = processed_row.get('metadata', {}).get('artist', 'Unknown Artist')
    features = processed_row.get('features', {})

    # Filter features: numeric lists, excluding masks
    piece_data = {
        k: v for k, v in features.items()
        if isinstance(v, list) and k not in ['Voice Mask', 'Guitar Mask'] and len(v) > 0
    }

    # Skip if insufficient features or inconsistent lengths
    if len(piece_data) < 2:
        # print(f"Skipping {title} by {artist}: fewer than 2 features.")
        continue

    lengths = {len(vals) for vals in piece_data.values()}
    if len(lengths) != 1:
        # print(f"Skipping {title} by {artist}: features have different lengths.")
        continue

    df_piece = pd.DataFrame(piece_data)
    n = len(df_piece)

    # Calculate block length for bootstrap (using SBB block_length logic)
    # Assuming default_block_length function is available from previous cells
    bl = default_block_length(n)

    # Run bootstrap for SBB (Stationary Block Bootstrap)
    # If bootstrap_corr_matrix function is not available, copy it here.
    # Using SBB as it is generally preferred for time series data with unknown dependence.
    med_df, lo_df, hi_df, p_df = bootstrap_corr_matrix(
        df_piece,
        B=BOOTSTRAP_B,
        ci=CI,
        threshold=THRESHOLD,
        absolute=ABSOLUTE,
        method="sbb", # Use SBB for this aggregation
        block_length=bl,
        random_state=RANDOM_STATE
    )

    labels = list(med_df.columns)

    # Iterate through unique feature pairs (upper triangle only)
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            feature1 = labels[i]
            feature2 = labels[j]

            median_corr = med_df.iat[i, j]
            ci_low = lo_df.iat[i, j]
            ci_high = hi_df.iat[i, j]
            prob_above_threshold = p_df.iat[i, j]

            classification = classify_correlation(ci_low, ci_high, THRESHOLD)

            all_correlations_data.append({
                'Artist': artist,
                'Title': title,
                'Feature1': feature1,
                'Feature2': feature2,
                'Median': median_corr,
                'CI_Low': ci_low,
                'CI_High': ci_high,
                'Prob_Above_Threshold': prob_above_threshold,
                'Classification': classification
            })

# Convert the aggregated data into a pandas DataFrame
df_all_correlations = pd.DataFrame(all_correlations_data)

print(f"Aggregated bootstrap results for {len(df_all_correlations)} feature pairs across all pieces.")
display(df_all_correlations.head())

**Reasoning**:
The previous step successfully created `df_all_correlations`, which contains the aggregated bootstrap results. The next step is to generate a scatter plot as described in the main task, visualizing the median correlation with confidence intervals and color-coding by classification.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define the threshold from the global variable
CORRELATION_THRESHOLD = THRESHOLD

plt.figure(figsize=(14, 8))

# Create a mapping for colors based on classification
color_map = {
    "stable strong": "red",
    "stable weak": "green",
    "borderline": "orange"
}

# Create the scatter plot with error bars
sns.scatterplot(
    data=df_all_correlations,
    x='Median',
    y='Feature1' + ' - ' + df_all_correlations['Feature2'],
    hue='Classification',
    palette=color_map,
    s=100,  # Size of the points
    zorder=2, # Ensure points are on top of error bars
    legend='full' # Show the legend
)

# Add error bars
plt.errorbar(
    x=df_all_correlations['Median'],
    y=df_all_correlations['Feature1'] + ' - ' + df_all_correlations['Feature2'],
    xerr=[df_all_correlations['Median'] - df_all_correlations['CI_Low'],
          df_all_correlations['CI_High'] - df_all_correlations['Median']],
    fmt='none', # Do not plot markers for error bars
    capsize=5,  # Length of the error bar caps
    color='gray', # Color of the error bars
    alpha=0.6, # Transparency
    zorder=1 # Ensure error bars are behind points
)

# Add a vertical line at the correlation threshold
plt.axvline(x=CORRELATION_THRESHOLD, color='blue', linestyle='--', linewidth=2, label=f'Threshold ({CORRELATION_THRESHOLD})')

# Customize plot labels and title
plt.xlabel('Median Bootstrap Correlation (|r|)', fontsize=14)
plt.ylabel('Feature Pair', fontsize=14)
plt.title('Feature Pair Correlations with 95% Bootstrap CI (Aggregated Across Pieces)', fontsize=16)
plt.xlim(-0.1, 1.1) # Set x-axis limits from -0.1 to 1.1 to accommodate correlations from 0 to 1
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Task
The plan is to extract phrase-related metrics (total phrases, distinct phrase labels, mean and standard deviation of phrase lengths in beats) for each song. These metrics will then be integrated into the existing DataFrame `df_final_raw`, which currently holds other song statistics, by merging on song metadata.

Here's the execution plan:
1.  **Extract Phrase Metrics**:
    *   Iterate through each song in the `data` (raw JSON) variable.
    *   For each song, retrieve the `phrase_labels`, `phrase_times`, and `phrase_durations` from the `'structural'` key, and `beat_times` from the `'rhythmic'` key.
    *   Calculate the total number of phrases (`len(phrase_labels)`).
    *   Calculate the count of distinct phrase labels (`len(set(phrase_labels))`).
    *   For each phrase, determine its start and end beat indices using `np.searchsorted` on the `beat_times` array. Calculate the length of the phrase in beats.
    *   Collect all phrase lengths in beats for the song.
    *   Calculate the mean and standard deviation of these phrase lengths. Handle cases where there are no phrases or only one phrase (std will be `np.nan`).
    *   Store these calculated metrics along with the song's `title`, `artist`, and `project_year` from its `metadata`.

2.  **Integrate into DataFrame**:
    *   Create a new Pandas DataFrame (`df_phrase_metrics`) from the collected song metadata and phrase metrics.
    *   Set the index of this new DataFrame to `['title', 'artist', 'year']` to match the existing `df_final_raw` DataFrame.
    *   Merge `df_final_raw` with `df_phrase_metrics` using a left join on their indices, adding the new phrase-related columns to `df_final_raw`.
    *   Display the updated `df_final_raw`.

## Extract Phrase Metrics

### Subtask:
For each song, retrieve phrase labels, start times, durations, and beat times. Calculate the total number of phrases, the count of distinct phrase labels, and the mean and standard deviation of phrase lengths in beats.


**Reasoning**:
To extract phrase metrics as requested, I will iterate through each song, gather metadata, structural data (phrase labels, times, durations), and rhythmic information (beat times). I'll calculate the total phrases, distinct phrase labels, and the mean/standard deviation of phrase lengths in beats using the provided logic, storing the results in a list of dictionaries.



In [ ]:
import numpy as np
import pandas as pd

phrase_metrics_list = []

for song in data:
    # 3a. Extract metadata
    title = song.get('metadata', {}).get('title', 'Unknown Title')
    artist = song.get('metadata', {}).get('artist', 'Unknown Artist')
    project_year_raw = song.get('metadata', {}).get('project_year', None)
    try:
        year = int(project_year_raw)
    except (ValueError, TypeError):
        year = np.nan

    # 3b. Retrieve structural data
    structural = song.get('structural', {})
    phrase_labels = structural.get('phrase_labels', [])
    phrase_times = structural.get('phrase_times', [])
    phrase_durations = structural.get('phrase_durations', [])

    # Ensure all structural lists are consistent and not empty for processing
    if not (len(phrase_labels) == len(phrase_times) == len(phrase_durations)) or not phrase_labels:
        # Skip if structural data is incomplete or empty
        continue

    # 3c. Retrieve beat times
    beat_times = np.array(song.get('rhythmic', {}).get('beat_times', []))

    if beat_times.size == 0:
        # Skip if beat_times are missing, as phrase lengths in beats cannot be calculated
        continue

    # 3d. Calculate total phrases
    total_phrases = len(phrase_labels)

    # 3e. Calculate distinct phrase labels by normalizing to the base letter
    normalized_phrase_labels = {label[0].upper() for label in phrase_labels if label and label[0].isalpha()}
    distinct_phrase_labels = len(normalized_phrase_labels)

    # 3f. Create an empty list for phrase lengths in beats
    phrase_lengths_in_beats = []

    # 3g. Iterate through phrase_times and phrase_durations
    for i in range(total_phrases):
        phrase_start_time = phrase_times[i]
        phrase_end_time = phrase_times[i] + phrase_durations[i]

        # 3g.ii. Find start and end beat indices
        # np.searchsorted returns the index where the value would be inserted to maintain order
        start_beat_index = np.searchsorted(beat_times, phrase_start_time, side='left')
        end_beat_index = np.searchsorted(beat_times, phrase_end_time, side='left')

        # 3g.iii. Calculate length in beats
        # If start_beat_index equals end_beat_index, it means the phrase duration was less than one beat or entirely between beats
        # In such cases, if it starts and ends within the same beat interval, length is 1 beat, otherwise 0.
        length_in_beats = end_beat_index - start_beat_index
        if length_in_beats == 0 and start_beat_index < len(beat_times) and beat_times[start_beat_index] > phrase_start_time:
            # If phrase starts before a beat and ends before the next, but within a beat interval, count as 1 beat if it's the start of a beat.
            length_in_beats = 1 # Smallest unit for a phrase, even if less than a full beat duration
        elif length_in_beats == 0 and phrase_durations[i] > 0:
            length_in_beats = 1 # if the duration is positive, even if it doesn't span beat marks perfectly.

        phrase_lengths_in_beats.append(length_in_beats)

    # 3h. Calculate mean and std of phrase lengths in beats
    mean_phrase_length_beats = np.nan
    std_phrase_length_beats = np.nan

    if phrase_lengths_in_beats:
        # Convert to numpy array to handle potential NaNs during calculations
        lengths_array = np.array(phrase_lengths_in_beats, dtype=float)
        mean_phrase_length_beats = np.mean(lengths_array)
        if len(lengths_array) > 1:
            std_phrase_length_beats = np.std(lengths_array)
        else:
            # If only one phrase length, std dev is 0. but np.nan for consistency with empty array.
            std_phrase_length_beats = 0.0 # or np.nan if you prefer, but 0.0 is more accurate for a single value

    # 3i. Append metrics to the list
    phrase_metrics_list.append({
        'Title': title,
        'Artist': artist,
        'Year': year,
        'Total Phrases': total_phrases,
        'Distinct Phrase Labels': distinct_phrase_labels,
        'Mean Phrase Length (beats)': mean_phrase_length_beats,
        'Std Phrase Length (beats)': std_phrase_length_beats
    })

# Convert the list of dictionaries to a pandas DataFrame
df_phrase_metrics = pd.DataFrame(phrase_metrics_list)

# Display the resulting DataFrame
display(df_phrase_metrics)
print(f"Successfully calculated phrase metrics for {len(df_phrase_metrics)} songs.")

**Reasoning**:
The `df_phrase_metrics` DataFrame has been created. The next step is to integrate it with `df_final_raw` by setting the index of `df_phrase_metrics` to match `df_final_raw` and then performing a left merge. This will add the new phrase-related metrics as columns to `df_final_raw`.



In [ ]:
import numpy as np
import pandas as pd

# Rename columns in df_phrase_metrics to match the index names of df_final_raw
df_phrase_metrics_renamed = df_phrase_metrics.rename(columns={
    'Title': 'title',
    'Artist': 'artist',
    'Year': 'year'
})

# Set the index of df_phrase_metrics_renamed to match df_final_raw
df_phrase_metrics_indexed = df_phrase_metrics_renamed.set_index(['title', 'artist', 'year'])

# Combine mean and std of phrase length into a single 'mean (std)' column
df_phrase_metrics_indexed['Phrase Length (beats)'] = df_phrase_metrics_indexed.apply(
    lambda row: f"{row['Mean Phrase Length (beats)']:.2f} ({row['Std Phrase Length (beats)']:.2f})", axis=1
)

# Drop the original separate mean and std columns
df_phrase_metrics_indexed = df_phrase_metrics_indexed.drop(columns=[
    'Mean Phrase Length (beats)',
    'Std Phrase Length (beats)'
])

# Merge df_final_raw with df_phrase_metrics_indexed using a left join
df_final_raw = df_final_raw.merge(
    df_phrase_metrics_indexed,
    left_index=True,
    right_index=True,
    how='left'
)

# Display the updated df_final_raw
print("Updated df_final_raw with phrase metrics:")
display(df_final_raw)

### Plotting Diachronic Evolution for Each Descriptor

This code iterates through each descriptor, plotting its yearly mean and standard deviation to visualize trends over time.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the 'year' column is available from the index for plotting
df_yearly_trends_plot = df_yearly_trends.reset_index()

# Get a list of unique base descriptor names (e.g., 'Gtr loudness' from 'Gtr loudness_mean')
descriptor_bases = sorted(list(set([col.replace('_mean', '').replace('_std', '') for col in descriptor_columns])))

plt.figure(figsize=(15, 5 * len(descriptor_bases)))
plt.suptitle('Diachronic Evolution of Musical Descriptors', fontsize=16, y=1.00)

for i, base_name in enumerate(descriptor_bases):
    mean_col = f'{base_name}_mean'
    std_col = f'{base_name}_std'

    # Check if both mean and std columns exist for the descriptor
    if mean_col in df_yearly_trends_plot.columns and std_col in df_yearly_trends_plot.columns:
        ax = plt.subplot(len(descriptor_bases), 1, i + 1)

        # Plot mean trend
        sns.lineplot(x='year', y=mean_col, data=df_yearly_trends_plot, marker='o', label='Mean', ax=ax)

        # Add shaded region for standard deviation
        ax.fill_between(
            df_yearly_trends_plot['year'],
            df_yearly_trends_plot[mean_col] - df_yearly_trends_plot[std_col],
            df_yearly_trends_plot[mean_col] + df_yearly_trends_plot[std_col],
            color='blue', alpha=0.2, label='Std Dev' # Blue shaded region
        )

        ax.set_title(f'Evolution of {base_name}', fontsize=14)
        ax.set_xlabel('Year', fontsize=12)
        ax.set_ylabel(base_name, fontsize=12)
        ax.legend(fontsize=10)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.set_xticks(df_yearly_trends_plot['year'])
        ax.tick_params(axis='x', rotation=45)

plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust rect to make space for suptitle
plt.show()

# Task
Preprocess `df_final_raw` by resetting its index and extracting numerical mean and standard deviation values from the string columns (e.g., 'Gtr loudness', 'Tonal dissonance') into new numerical columns for each feature.

## Preprocess Data for Diachronic Analysis

### Subtask:
Reset the index of `df_final_raw` and extract numerical mean and standard deviation values from the 'mean (std)' string columns for each descriptor, creating new numerical columns.


**Reasoning**:
The subtask requires resetting the index of `df_final_raw` and then parsing specific string columns containing 'mean (std)' values into separate numerical mean and standard deviation columns. This involves regex extraction and type conversion.



In [ ]:
import numpy as np
import pandas as pd
import re

# 1. Reset the index of df_final_raw
df_final_raw = df_final_raw.reset_index()

# Identify columns with the 'mean (std)' string format
# We assume these are numeric-like columns that have been formatted as strings
# and exclude known non-feature columns like 'title', 'artist', 'year',
# 'Total Phrases', 'Distinct Phrase Labels'.

# Get all columns except the metadata and integer count columns
columns_to_process = [col for col in df_final_raw.columns if col not in ['title', 'artist', 'year', 'Total Phrases', 'Distinct Phrase Labels']]

# Exclude 'Phrase Length (beats)' from this initial list as it's handled separately
# in the previous cell and has a similar format, but its components are already numeric.
# We will add it back explicitly if needed, but for now focus on the original 'mean_std' features

for col in columns_to_process:
    # Only process if the column is of string type and contains the pattern
    # Check the first non-null value to determine if it matches the pattern
    first_valid_value = df_final_raw[col].dropna().iloc[0] if not df_final_raw[col].dropna().empty else None

    if isinstance(first_valid_value, str) and re.match(r'^-?\d+\.\d+ \(-?\d+\.\d+\)$', first_valid_value):
        print(f"Processing column: {col}")
        # Extract mean and std values
        df_final_raw[f'{col}_mean'] = df_final_raw[col].apply(lambda x: float(re.match(r'^(-?\d+\.\d+)', x).group(1)) if pd.notna(x) else np.nan)
        df_final_raw[f'{col}_std'] = df_final_raw[col].apply(lambda x: float(re.search(r'\((-?\d+\.\d+)\)', x).group(1)) if pd.notna(x) else np.nan)

        # Drop the original column
        df_final_raw = df_final_raw.drop(columns=[col])


# Handle the 'Phrase Length (beats)' column separately if it exists and needs parsing
# It was created in a similar 'mean (std)' format, so the same logic applies.
if 'Phrase Length (beats)' in df_final_raw.columns:
    col = 'Phrase Length (beats)'
    first_valid_value = df_final_raw[col].dropna().iloc[0] if not df_final_raw[col].dropna().empty else None

    if isinstance(first_valid_value, str) and re.match(r'^-?\d+\.\d+ \(-?\d+\.\d+\)$', first_valid_value):
        print(f"Processing column: {col}")
        df_final_raw[f'{col}_mean'] = df_final_raw[col].apply(lambda x: float(re.match(r'^(-?\d+\.\d+)', x).group(1)) if pd.notna(x) else np.nan)
        df_final_raw[f'{col}_std'] = df_final_raw[col].apply(lambda x: float(re.search(r'\((-?\d+\.\d+)\)', x).group(1)) if pd.notna(x) else np.nan)
        df_final_raw = df_final_raw.drop(columns=[col])


# Display the head of the updated DataFrame
print("\nUpdated df_final_raw with numerical mean and std columns:")
display(df_final_raw.head())

## Aggregate Descriptors by Year

### Subtask:
Group the preprocessed data by 'year' and calculate the mean of song-level mean and standard deviation values for each descriptor.


**Reasoning**:
First, I'll identify all columns that end with `_mean` or `_std` in the `df_final_raw` DataFrame, excluding the `Total Phrases` and `Distinct Phrase Labels` as they are counts and not feature descriptors.



In [ ]:
import numpy as np
import pandas as pd

# Identify columns ending with '_mean' or '_std'
# Exclude 'Total Phrases_mean', 'Total Phrases_std', 'Distinct Phrase Labels_mean', 'Distinct Phrase Labels_std'
# because these are counts, not features for diachronic analysis in the same way.

descriptor_columns = [col for col in df_final_raw.columns if
                      (col.endswith('_mean') or col.endswith('_std')) and
                      not (col.startswith('Total Phrases') or col.startswith('Distinct Phrase Labels'))
                     ]

# Group by 'year' and calculate the mean of the identified columns
df_yearly_trends = df_final_raw.groupby('year')[descriptor_columns].mean()

# Display the resulting DataFrame
print("Yearly aggregated trends and variability for each descriptor:")
display(df_yearly_trends)

## Generate Diachronic Evolution Plots

### Subtask:
For each descriptor, create a line plot with 'year' on the x-axis and the yearly aggregated mean on the y-axis, adding error bars (or shaded regions) using the yearly aggregated standard deviation.


## Summary:

### Data Analysis Key Findings

*   The `df_final_raw` DataFrame was successfully preprocessed by resetting its index, making 'title', 'artist', and 'year' regular columns.
*   String-formatted columns containing 'mean (std)' patterns, such as 'Gtr loudness', 'Tonal dissonance', and 'Phrase Length (beats)', were identified and transformed. For each of these, two new numerical columns were created: one for the mean (suffixed with `_mean`) and one for the standard deviation (suffixed with `_std`).
*   The extracted values were successfully converted to float data type, and the original string columns were subsequently dropped from the DataFrame.
*   A new DataFrame, `df_yearly_trends`, was created by grouping the preprocessed data by 'year'. For each year, the mean of the song-level `_mean` and `_std` values of 22 relevant descriptors (e.g., 'Gtr loudness\_mean', 'Tonal dissonance\_std') was calculated, excluding count-based features like 'Total Phrases' and 'Distinct Phrase Labels'.

### Insights or Next Steps

*   The `df_yearly_trends` DataFrame, now containing the yearly aggregated mean and standard deviation for various musical descriptors, is perfectly structured for generating diachronic evolution plots.
*   The successful extraction and numerical conversion of feature statistics from string representations enable robust time-series analysis to observe trends and variability of musical characteristics over the years.
